In [ ]:
# ============================================================
# Skieur End-to-End Lie Dynamics -- Config + Imports
# ============================================================
# Jointly trains encoder + Lie generator (end-to-end) instead of the
# two-stage CEBRA -> OLS pipeline.
#
# This is an exploratory behavioural-constrained latent dynamics
# modelling framework.  Default transition is discrete matrix_exp
# (not continuous ODE); USE_ODE is experimental.
#
# Key features over baseline:
#   1. End-to-end joint training: Loss = L_InfoNCE + lambda * L_dynamics
#   2. Strict Lie parameterization: skew basis G_i + matrix_exp
#   3. Variance-normalized dynamics loss (prevents ||z||->0 trivial solution)
#   4. Nonlinear multidim forward model: J(t) = sum_i w_i(u_t) G_i

import os, sys, gc, json, datetime, warnings
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from scipy.stats import ttest_rel
from scipy.ndimage import gaussian_filter
import seaborn as sns
import pickle
from tqdm.auto import tqdm, trange
from collections import defaultdict
import copy

# --- Core DL ---
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# --- Existing config (reused from Skieur_LieAlgebra_CEBRA.ipynb) ---
dt = 0.005
t_pre, t_post = 0.3, 0.3
SESSION_TYPE = "playback"
USE_MACRO_EPOCH = True
CEBRA_DISTANCE = "euclidean"
CEBRA_ARCH = "offset10-model" if CEBRA_DISTANCE == "cosine" else "offset10-model-mse"
CEBRA_EMBEDDING_DIM = 3
TAU_SHIFT = 6
LIE_METHOD = "lstsq"
MIN_EPOCH_DUR = 2.0
NAS = r"\\129.199.81.18\data5\eTheremin"

# --- New config for end-to-end pipeline ---
D_LATENT = 3                    # latent dimension (3/6/8)
USE_ODE = False                 # True = torchdiffeq.odeint; False = discrete matrix_exp
ODE_METHOD = "rk4"              # "rk4" or "dopri5" (only if USE_ODE=True; experimental)
LAMBDA_DYN = 0.1                # dynamics loss weight
LAMBDA_DYN_WARMUP = 200         # steps of lambda=0 warmup before ramping
CONSTRAINED_L = False           # True: L = -C@C.T (stable dissipation); False: unconstrained
DRIVE_KEYS = ["Velocity_x"]     # drive features (extendable)
ENCODER_HIDDEN = [128, 64]      # encoder hidden channel sizes
CONTROL_HIDDEN = [32]           # control net hidden sizes
MINI_TRAJ_LEN = 20              # mini-trajectory length in bins (~100ms at dt=0.005)
VAL_ROLLOUT_LENS = [20, 50, 100]  # multi-scale validation windows (bins; 100ms/250ms/500ms)
N_EPOCHS_TRAIN = 50             # outer training epochs (per session)
BATCH_SIZE = 512                # frames per batch (flattened)
LR = 3e-4                       # learning rate
WEIGHT_DECAY = 1e-6             # AdamW weight decay
GRAD_CLIP = 1.0                 # gradient clipping norm
TEMPERATURE = 1.5               # InfoNCE temperature (matched to CEBRA baseline for fair comparison)
MIN_EPOCH_TIMEPOINTS = 200      # minimum timepoints per epoch
MIN_EPOCHS_PER_COND = 1         # minimum epochs per condition (E2E internally requires >=2 for train/val)
N_SHUFFLES = 50                 # shuffle realizations (screening null; >=500 for formal inference)
TRAIN_VAL_SPLIT = 0.8           # fraction of epochs for training (remainder held-out)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
RANDOM_SEED = 42                # base seed for reproducibility
N_TRAIN_SESSIONS = None         # None = use all available sessions
N_SEEDS = 3                     # number of random seeds per session (report seed variance)

# --- Reproducibility ---
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

# --- HAS_* flags for optional dependencies ---
HAS_CEBRA = False
try:
    from cebra import CEBRA
    HAS_CEBRA = True
    print("cebra: OK")
except ImportError:
    print("cebra: NOT INSTALLED -- Dummy-CEBRA control will be skipped")

HAS_TORCHDIFFEQ = False
try:
    import torchdiffeq
    HAS_TORCHDIFFEQ = True
    print("torchdiffeq: OK")
except ImportError:
    print("torchdiffeq: NOT INSTALLED -- ODE mode disabled (discrete matrix_exp fallback)")

HAS_GEOOPT = False
try:
    import geoopt
    HAS_GEOOPT = True
    print("geoopt: OK (not used by default)")
except ImportError:
    print("geoopt: not installed (optional, not required)")

# --- Matplotlib style ---
mpl.rcdefaults()
plt.rcParams.update({
    'font.size': 7, 'axes.linewidth': 0.5,
    'axes.spines.top': False, 'axes.spines.right': False,
    'xtick.major.width': 0.5, 'ytick.major.width': 0.5,
    'xtick.major.size': 2, 'ytick.major.size': 2,
    'xtick.direction': 'out', 'ytick.direction': 'out',
    'pdf.fonttype': 42, 'ps.fonttype': 42,
})
warnings.filterwarnings("ignore", category=FutureWarning)

print(f"Device: {DEVICE}")
print(f"D_LATENT={D_LATENT}, USE_ODE={USE_ODE}, LAMBDA_DYN={LAMBDA_DYN}")
print(f"DRIVE_KEYS={DRIVE_KEYS}, CONSTRAINED_L={CONSTRAINED_L}")
print(f"TEMPERATURE={TEMPERATURE}, N_SHUFFLES={N_SHUFFLES}, N_SEEDS={N_SEEDS}")
print(f"VAL_ROLLOUT_LENS={VAL_ROLLOUT_LENS} bins (multi-scale)")
print(f"N_TRAIN_SESSIONS={'all' if N_TRAIN_SESSIONS is None else N_TRAIN_SESSIONS}")
print("Cell 0 -- Config ready.")

In [ ]:
# ============================================================
# Load spike-sorted Skieur data
# ============================================================

def load_pickled_ss(file_prefix, session_type, dt):
    """Load pickled spike-sorted data from NAS."""
    data_path = os.path.join(NAS, f"{file_prefix}_{session_type}_{dt}_data_ss")
    feat_path = os.path.join(NAS, f"{file_prefix}_{session_type}_{dt}_feature_ss")
    with open(data_path, "rb") as f:
        n_data = pickle.load(f)
    with open(feat_path, "rb") as f:
        f_data = pickle.load(f)
    return n_data, f_data


print("Loading hs0...")
n_data_hs0, f_data_hs0 = load_pickled_ss("SKIEUR_hs_0", SESSION_TYPE, dt)
print(f"  hs0: {len(n_data_hs0)} sessions")

print("Loading hs1...")
n_data_hs1, f_data_hs1 = load_pickled_ss("SKIEUR_hs_1", SESSION_TYPE, dt)
print(f"  hs1: {len(n_data_hs1)} sessions")

# Add Velocity_x
for f_df in f_data_hs0 + f_data_hs1:
    pos = f_df["Position"].values
    vel = np.diff(pos); vel = np.append(0, vel)
    vel = vel * 100; vel[~np.isfinite(vel)] = 0
    f_df["Velocity_x"] = vel

n_data_all_raw = list(n_data_hs0) + list(n_data_hs1)
f_data_all_raw = list(f_data_hs0) + list(f_data_hs1)
n_hs0 = len(n_data_hs0)

MIN_GC = 10
n_data_all, f_data_all = [], []
n_hs0_filtered = 0
for i, nd in enumerate(n_data_all_raw):
    if nd.shape[0] >= MIN_GC:
        n_data_all.append(nd)
        f_data_all.append(f_data_all_raw[i])
        if i < n_hs0:
            n_hs0_filtered += 1

n_hs0 = n_hs0_filtered
print(f"After gc>={MIN_GC}: {len(n_data_all)} sessions "
      f"(hs0={n_hs0}, hs1={len(n_data_all)-n_hs0})")

example = f_data_all[0]
print(f"Example: {example.shape[0]:,} tp, {n_data_all[0].shape[0]} neurons")
print(f"  Velocity_x: [{example['Velocity_x'].min():.1f}, "
      f"{example['Velocity_x'].max():.1f}]")
print(f"  Position:   [{example['Position'].min():.1f}, "
      f"{example['Position'].max():.1f}]")
print("Cell 1 -- Data loaded.")

In [ ]:
# ============================================================
# Helper functions: epoch extraction + Lie algebra + new utils
# ============================================================

# --- Reused from existing notebook ---

def preprocess_data(data_list, method="l2"):
    """Preprocess list of (time, neurons) arrays.
    method='l2': per-timepoint L2 normalization (for cosine distance)
    method='zscore': per-neuron Z-score across time (for euclidean distance)
    """
    out = []
    for d in data_list:
        if method == "zscore":
            mean = np.mean(d, axis=0, keepdims=True)
            std = np.std(d, axis=0, keepdims=True)
            std[std == 0] = 1e-9
            out.append(((d - mean) / std).astype(np.float32))
        else:  # l2
            norms = np.linalg.norm(d, axis=1, keepdims=True)
            norms[norms == 0] = 1e-9
            out.append((d / norms).astype(np.float32))
    return out


def extract_macro_epochs(n_data_session, f_df, condition_val, dt,
                          min_duration=2.0, label_col="Velocity_x"):
    """Extract contiguous macro-epochs of the same Condition."""
    conditions = f_df["Condition"].values
    mask = (conditions == condition_val)
    min_bins = int(min_duration / dt)
    epochs_n, epochs_l = [], []
    in_epoch, start = False, 0
    for i in range(len(mask)):
        if mask[i] and not in_epoch:
            start = i; in_epoch = True
        elif not mask[i] and in_epoch:
            if i - start >= min_bins:
                epochs_n.append(n_data_session[:, start:i].T.astype(np.float32))
                epochs_l.append(f_df[label_col].values[start:i].astype(np.float32))
            in_epoch = False
    if in_epoch and (len(mask) - start) >= min_bins:
        epochs_n.append(n_data_session[:, start:].T.astype(np.float32))
        epochs_l.append(f_df[label_col].values[start:].astype(np.float32))
    return epochs_n, epochs_l, len(epochs_n)


def extract_epochs(n_data_session, f_df, condition_val, dt,
                   label_col="Velocity_x", step=1):
    """Unified extraction: macro-epochs or peri-event windows."""
    if USE_MACRO_EPOCH:
        epochs_n, epochs_l, n_ep = extract_macro_epochs(
            n_data_session, f_df, condition_val, dt,
            min_duration=MIN_EPOCH_DUR, label_col=label_col)
        if step > 1:
            epochs_n = [e[::step].astype(np.float32) for e in epochs_n]
            epochs_l = [l[::step].astype(np.float32) for l in epochs_l]
        method = "l2" if CEBRA_DISTANCE == "cosine" else "zscore"
        epochs_n = preprocess_data(epochs_n, method=method)
        if TAU_SHIFT > 0:
            epochs_n, epochs_l = zip(*[(e_n[TAU_SHIFT:], e_l[:-TAU_SHIFT])
                                        for e_n, e_l in zip(epochs_n, epochs_l)])
            epochs_n, epochs_l = list(epochs_n), list(epochs_l)
        return epochs_n, epochs_l, n_ep
    else:
        trigger_mask = ((f_df["Condition"].values == condition_val) &
                        (f_df["Frequency_changes"].values == 1))
        trigger_indices = np.where(trigger_mask)[0]
        n_pre = int(t_pre / dt)
        n_post = int(t_post / dt)
        windows_n, windows_l = [], []
        for idx in trigger_indices:
            start = idx - n_pre
            end = idx + n_post + 1
            if start >= 0 and end <= n_data_session.shape[1]:
                windows_n.append(n_data_session[:, start:end].T.astype(np.float32))
                windows_l.append(f_df[label_col].values[start:end].astype(np.float32))
        if step > 1:
            windows_n = [w[::step].astype(np.float32) for w in windows_n]
            windows_l = [l[::step].astype(np.float32) for l in windows_l]
        method = "l2" if CEBRA_DISTANCE == "cosine" else "zscore"
        windows_n = preprocess_data(windows_n, method=method)
        if TAU_SHIFT > 0:
            windows_n, windows_l = zip(*[(w_n[TAU_SHIFT:], w_l[:-TAU_SHIFT])
                                          for w_n, w_l in zip(windows_n, windows_l)])
            windows_n, windows_l = list(windows_n), list(windows_l)
        return windows_n, windows_l, len(windows_n)


def _fit_lie_lstsq(r, x_dot, dt=0.005):
    """OLS + skew-symmetrization (original baseline method)."""
    T, N = r.shape
    dr_dt = np.gradient(r, dt, axis=0)
    U_rot = r * x_dot[:, np.newaxis]
    U = np.hstack([U_rot, r])
    weights_T, _, _, _ = np.linalg.lstsq(U, dr_dt, rcond=None)
    weights = weights_T.T
    J_ols = weights[:, :N]
    J_skew = 0.5 * (J_ols - J_ols.T)
    norm_total = np.linalg.norm(J_ols)
    norm_skew = np.linalg.norm(J_skew)
    sr = norm_skew / norm_total if norm_total > 1e-9 else 0
    dR_pred = U_rot @ J_skew.T + r @ weights[:, N:].T
    ss_res = np.sum((dr_dt - dR_pred) ** 2)
    ss_tot = np.sum((dr_dt - np.mean(dr_dt)) ** 2)
    r2 = 1 - ss_res / ss_tot if ss_tot > 1e-9 else 0
    dR_leak = r @ weights[:, N:].T
    ss_leak = np.sum((dr_dt - dR_leak) ** 2)
    r2_drive = 1 - ss_res / ss_leak if ss_leak > 1e-9 else 0
    return J_skew, sr, r2, J_ols, r2_drive


def _fit_lie_pytorch(r, x_dot, dt=0.005, n_iter=500, lr=1e-3):
    """Constrained optimization: J_skew = W - W^T via PyTorch."""
    T, N = r.shape
    dr_dt = np.gradient(r, dt, axis=0)
    R_t = torch.tensor(r, dtype=torch.float32)
    X_t = torch.tensor(x_dot, dtype=torch.float32).reshape(-1, 1)
    dR_t = torch.tensor(dr_dt, dtype=torch.float32)
    W = torch.zeros(N, N, requires_grad=True)
    L_t = torch.zeros(N, N, requires_grad=True)
    opt = torch.optim.Adam([W, L_t], lr=lr)
    for _ in range(n_iter):
        opt.zero_grad()
        J_t = W - W.T
        dR_pred = (R_t * X_t) @ J_t.T + R_t @ L_t.T
        loss = torch.mean((dR_t - dR_pred) ** 2)
        loss.backward()
        opt.step()
    with torch.no_grad():
        J_skew = (W - W.T).numpy()
        L_np = L_t.numpy()
    U = np.hstack([r * x_dot[:, None], r])
    w_T, _, _, _ = np.linalg.lstsq(U, dr_dt, rcond=None)
    J_ols = w_T.T[:, :N]
    sr = (np.linalg.norm(J_skew) / np.linalg.norm(J_ols)
          if np.linalg.norm(J_ols) > 1e-9 else 0)
    dR_pred = (r * x_dot[:, None]) @ J_skew.T + r @ L_np.T
    ss_res = np.sum((dr_dt - dR_pred) ** 2)
    ss_tot = np.sum((dr_dt - np.mean(dr_dt)) ** 2)
    r2 = 1 - ss_res / ss_tot if ss_tot > 1e-9 else 0
    dR_leak = r @ L_np.T
    ss_leak = np.sum((dr_dt - dR_leak) ** 2)
    r2_drive = 1 - ss_res / ss_leak if ss_leak > 1e-9 else 0
    return J_skew, sr, r2, J_ols, r2_drive


def fit_lie_algebra_with_leak(r, x_dot, dt=0.005, n_iter=500, lr=1e-3):
    """Fit dR/dt = J_skew * R * x_dot + L * R.
    Method controlled by global LIE_METHOD.
    """
    if LIE_METHOD == "lstsq":
        return _fit_lie_lstsq(r, x_dot, dt)
    else:
        return _fit_lie_pytorch(r, x_dot, dt, n_iter, lr)


# --- NEW helper functions ---

def build_drive_vector(f_df, drive_keys=None):
    """Build multi-dimensional drive vector from feature DataFrame."""
    if drive_keys is None:
        drive_keys = DRIVE_KEYS
    components = []
    for key in drive_keys:
        if key in f_df.columns:
            x = f_df[key].values.astype(np.float32)
            mu, std = np.mean(x), np.std(x)
            if std < 1e-9:
                std = 1.0
            components.append((x - mu) / std)
        else:
            print(f"  WARNING: drive key '{key}' not in f_df; skipping")
    if not components:
        x = f_df["Velocity_x"].values.astype(np.float32)
        mu, std = np.mean(x), np.std(x)
        if std < 1e-9:
            std = 1.0
        components.append((x - mu) / std)
    return np.column_stack(components) if len(components) > 1 else components[0]


def create_mini_trajectories(epochs_n, epochs_l, drive_epochs,
                              traj_len=20, n_samples_per_epoch=50):
    """Sample mini-trajectories from epoch data for joint training.

    Each mini-trajectory is a contiguous window of length traj_len,
    sampled uniformly from within the epoch.

    Returns (traj_n, traj_l, traj_d) or (None, None, None) if no valid data.
    """
    all_n, all_l, all_d = [], [], []
    for e_n, e_l, e_d in zip(epochs_n, epochs_l, drive_epochs):
        T_ep = e_n.shape[0]
        if T_ep < traj_len:
            continue
        n_sample = min(n_samples_per_epoch, T_ep - traj_len + 1)
        starts = np.random.choice(T_ep - traj_len + 1, size=n_sample,
                                  replace=False)
        for s in starts:
            all_n.append(e_n[s:s + traj_len])
            all_l.append(e_l[s:s + traj_len] if e_l.ndim == 1
                         else e_l[s:s + traj_len])
            all_d.append(e_d[s:s + traj_len] if e_d.ndim == 2
                         else e_d[s:s + traj_len, None])
    if not all_n:
        return None, None, None
    return np.stack(all_n), np.stack(all_l), np.stack(all_d)


def compute_eigenvalue_metrics(J_full):
    """Compute |Real| and |Imag| from eigenvalues of J_full.

    Returns (mean_abs_real, mean_abs_imag).
    """
    eigvals = np.linalg.eigvals(J_full)
    real_mean = np.mean(np.abs(np.real(eigvals)))
    imag_mean = np.mean(np.abs(np.imag(eigvals)))
    return real_mean, imag_mean


def info_nce_loss(z, labels, temperature=None):
    """InfoNCE loss for contrastive learning on continuous behavioral labels.

    For each anchor, finds k nearest neighbors in label space as positives.
    Uses per-row k-nearest (not global threshold) to avoid collapse from
    self-pair zeros dominating kthvalue on the full flattened matrix.
    """
    if temperature is None:
        temperature = TEMPERATURE  # from Cell 0 config
    z_norm = F.normalize(z, p=2, dim=1)
    sim = z_norm @ z_norm.T / temperature       # (N, N)
    label_diff = torch.abs(labels[:, None] - labels[None, :])  # (N, N)
    N = len(labels)
    k_pos = max(2, int(0.05 * N))

    # Per-row: exclude self (diagonal) and find k-th smallest label difference
    diff_no_self = label_diff + torch.eye(N, device=labels.device) * 1e10
    kth_per_row, _ = torch.kthvalue(diff_no_self, k_pos, dim=1)  # (N,)
    pos_mask = label_diff < kth_per_row[:, None]    # (N, N)
    pos_mask.fill_diagonal_(False)                   # still exclude self

    exp_sim = torch.exp(sim)
    pos_sum = (exp_sim * pos_mask.float()).sum(dim=1)           # (N,)
    all_sum = exp_sim.sum(dim=1) - exp_sim.diagonal()           # (N,) exclude self
    loss = -torch.log(pos_sum / (all_sum + 1e-9) + 1e-9).mean()
    return loss


print("Helpers ready: extract_epochs, fit_lie_algebra_with_leak, "
      "build_drive_vector, create_mini_trajectories, info_nce_loss")

## 1. Baseline Two-Stage Pipeline + Dummy-CEBRA Control

Re-run the existing two-stage CEBRA -> OLS-Lie pipeline for reference,
then add the missing **Dummy-CEBRA negative control** (train CEBRA on shuffled labels).

**Key comparison:**
- `SR_true` / `R2_drive_true`: true labels, real embedding -> genuine rotational structure
- `SR_dummy` / `R2_drive_dummy`: shuffled labels, dummy embedding -> InfoNCE artifact baseline
- **Gate:** `R2_drive_true > R2_drive_dummy` signifies rotational structure is a genuine property
  of neural population dynamics, not an artifact of InfoNCE imprinting rotational topology.

In [ ]:
# ============================================================
# Phase 0 -- Baseline Two-Stage + Dummy-CEBRA Control
# ============================================================
# For each session:
#   1. Train pooled CEBRA on true labels
#   2. Train dummy CEBRA on permuted labels (key negative control)
#   3. Per epoch: transform -> fit Lie -> compute SR/R2/R2_drive
#   4. Per epoch: shuffle label -> fit Lie -> compute null metrics
#   5. Per epoch: dummy embed -> fit Lie -> compute dummy metrics

CEBRA_LIE_ITERS = 3000
LIE_OUTPUT_DIR = f"Skieur_LieE2E_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}"
os.makedirs(LIE_OUTPUT_DIR, exist_ok=True)
print(f"Output directory: {LIE_OUTPUT_DIR}")

baseline_results = []  # per session-condition aggregate
dummy_results = []     # per session-condition for dummy CEBRA

if HAS_CEBRA:
    n_skipped_epochs, n_skipped_cond, n_sessions_used = 0, 0, 0

    for idx, (n_data_session, f_df) in enumerate(
            zip(tqdm(n_data_all, desc="Baseline+Dummy"), f_data_all)):
        hs_label = "hs0" if idx < n_hs0 else "hs1"

        # ---- Extract + filter epochs for BOTH conditions ----
        cond_epochs = {}
        cond_labels = {}
        skip_session = False
        for val, label in [(0.0, "Tracking"), (1.0, "Playback")]:
            epochs_n, epochs_l, n_ep = extract_epochs(
                n_data_session, f_df, val, dt, label_col="Velocity_x")
            # Filter short epochs
            valid_idx = [i for i in range(len(epochs_n))
                         if epochs_n[i].shape[0] >= MIN_EPOCH_TIMEPOINTS]
            n_skipped_epochs += len(epochs_n) - len(valid_idx)
            if len(valid_idx) < MIN_EPOCHS_PER_COND:
                skip_session = True
                n_skipped_cond += 1
                break
            cond_epochs[label] = [epochs_n[i] for i in valid_idx]
            cond_labels[label] = [epochs_l[i] for i in valid_idx]

        if skip_session:
            continue
        n_sessions_used += 1

        # ---- Pool epochs across conditions ----
        all_epochs = cond_epochs["Tracking"] + cond_epochs["Playback"]
        all_labels = cond_labels["Tracking"] + cond_labels["Playback"]
        n_track_ep = len(cond_epochs["Tracking"])

        # ---- Train ONE pooled CEBRA (true labels) ----
        try:
            cebra_true = CEBRA(
                model_architecture=CEBRA_ARCH,
                output_dimension=CEBRA_EMBEDDING_DIM,
                max_iterations=CEBRA_LIE_ITERS, batch_size=2048,
                learning_rate=3e-4, temperature=1.5,
                distance=CEBRA_DISTANCE,
                conditional="time_delta", device="cuda", verbose=False)
            cebra_true.fit(all_epochs, all_labels)
            loss_true = float(cebra_true.state_dict_['loss'][-1])
        except Exception as e:
            print(f"  Session {idx} CEBRA true failed: {e}")
            continue

        # ---- Train dummy CEBRA (shuffled labels) ----
        # NOTE: uses full permutation (destroys autocorrelation). This is a
        # "strong negative control" — destroys ALL label structure. For an
        # autocorrelation-preserving dummy, use circular-shift labels instead.
        all_labels_shuf = [np.random.permutation(l) for l in all_labels]
        cebra_dummy = None
        try:
            cebra_dummy = CEBRA(
                model_architecture=CEBRA_ARCH,
                output_dimension=CEBRA_EMBEDDING_DIM,
                max_iterations=CEBRA_LIE_ITERS, batch_size=2048,
                learning_rate=3e-4, temperature=1.5,
                distance=CEBRA_DISTANCE,
                conditional="time_delta", device="cuda", verbose=False)
            cebra_dummy.fit(all_epochs, all_labels_shuf)
        except Exception as e:
            print(f"  Session {idx} Dummy CEBRA failed: {e}")

        # ---- Per-condition evaluation ----
        for cond_name, start_idx in [("Tracking", 0),
                                      ("Playback", n_track_ep)]:
            n_ep = (n_track_ep if cond_name == "Tracking"
                    else len(all_epochs) - n_track_ep)
            ep_list = all_epochs[start_idx:start_idx + n_ep]
            lab_list = all_labels[start_idx:start_idx + n_ep]

            sr_vals, r2_vals, r2d_vals = [], [], []
            sr_sh_vals, r2_sh_vals, r2d_sh_vals = [], [], []
            eig_real_vals, eig_imag_vals = [], []
            sr_dummy_vals, r2d_dummy_vals = [], []

            for ei, (ep, el) in enumerate(zip(ep_list, lab_list)):
                # --- True embedding ---
                emb = cebra_true.transform(ep, session_id=start_idx + ei)
                J_s, sr, r2, J_ols, r2d = fit_lie_algebra_with_leak(emb, el)
                sr_vals.append(sr)
                r2_vals.append(r2)
                r2d_vals.append(r2d)
                re, im = compute_eigenvalue_metrics(J_ols)
                eig_real_vals.append(re)
                eig_imag_vals.append(im)

                # --- Shuffle control (same embedding, circular-shifted labels) ---
                # Uses np.roll to preserve autocorrelation structure (cf. E2E null
                # and lie_algebra_method_description.md section 12.3).
                T_lab = len(el)
                min_shift = max(1, MINI_TRAJ_LEN)
                s_sr, s_r2, s_r2d = [], [], []
                for _ in range(N_SHUFFLES):
                    shift = np.random.randint(min_shift, max(min_shift + 1, T_lab - min_shift))
                    el_sh = np.roll(el, shift)
                    _, sr_sh, r2_sh, _, r2d_sh = fit_lie_algebra_with_leak(
                        emb, el_sh)
                    s_sr.append(sr_sh)
                    s_r2.append(r2_sh)
                    s_r2d.append(r2d_sh)
                sr_sh_vals.append(np.mean(s_sr))
                r2_sh_vals.append(np.mean(s_r2))
                r2d_sh_vals.append(np.mean(s_r2d))

                # --- Dummy embedding (shuffled-label CEBRA) ---
                if cebra_dummy is not None:
                    emb_dummy = cebra_dummy.transform(
                        ep, session_id=start_idx + ei)
                    _, sr_d, _, _, r2d_d = fit_lie_algebra_with_leak(
                        emb_dummy, el)
                    sr_dummy_vals.append(sr_d)
                    r2d_dummy_vals.append(r2d_d)

            n_ep_valid = len(sr_vals)
            baseline_results.append({
                "Subject": "SKIEUR", "Session_Idx": idx,
                "Headstage": hs_label, "Condition": cond_name,
                "Space": "CEBRA", "N_Epochs": n_ep_valid,
                "N_Neurons": n_data_session.shape[0],
                "SR": np.mean(sr_vals), "SR_shuffle": np.mean(sr_sh_vals),
                "R2": np.mean(r2_vals), "R2_shuffle": np.mean(r2_sh_vals),
                "R2_drive": np.mean(r2d_vals),
                "R2_drive_shuffle": np.mean(r2d_sh_vals),
                "Eig_Real_Mean": np.mean(eig_real_vals),
                "Eig_Imag_Mean": np.mean(eig_imag_vals),
            })
            if sr_dummy_vals:
                dummy_results.append({
                    "Subject": "SKIEUR", "Session_Idx": idx,
                    "Headstage": hs_label, "Condition": cond_name,
                    "N_Epochs": n_ep_valid,
                    "SR_dummy": np.mean(sr_dummy_vals),
                    "R2_drive_dummy": np.mean(r2d_dummy_vals),
                })

    baseline_df = pd.DataFrame(baseline_results)
    dummy_df = pd.DataFrame(dummy_results) if dummy_results else None
    print(f"Baseline: {len(baseline_df)} session-conditions "
          f"({n_sessions_used} sessions used, {n_skipped_cond} skipped)")
    if dummy_df is not None:
        print(f"Dummy CEBRA: {len(dummy_df)} entries")

else:
    print("CEBRA not installed. Skipping baseline + dummy control.")
    baseline_df, dummy_df = None, None

### Baseline vs Dummy-CEBRA Quick Check

If rotational structure is genuine (not an InfoNCE artifact):
- `SR_true > SR_dummy` (true embedding has more rotational structure)
- `R2_drive_true >> R2_drive_dummy` (drive explains variance only when labels are real)
- `R2_drive_dummy ~ 0` (random labels provide no predictive power for dynamics)

This is the key negative control that the original notebook acknowledges is missing.

In [ ]:
# Quick baseline vs dummy comparison
if baseline_df is not None:
    print("=" * 60)
    print("  Baseline CEBRA-Embedded Lie -- Quick Summary")
    print("=" * 60)
    grp = baseline_df.groupby("Condition")[
        ["SR", "SR_shuffle", "R2", "R2_shuffle", "R2_drive",
         "R2_drive_shuffle"]].mean().round(4)
    print(grp.to_string())
    print()

    # Paired t-test: Tracking vs Playback
    pivot = baseline_df.pivot_table(
        values=["SR", "R2_drive"],
        index=["Subject", "Session_Idx", "Headstage"],
        columns="Condition").dropna()
    if len(pivot) > 1:
        for metric in ["SR", "R2_drive"]:
            t, p = ttest_rel(pivot[metric]["Tracking"],
                             pivot[metric]["Playback"])
            print(f"  {metric} Tracking vs Playback: t={t:.3f}, p={p:.4f}")

    if dummy_df is not None:
        merged = baseline_df.merge(dummy_df,
                                   on=["Subject", "Session_Idx",
                                       "Headstage", "Condition"])
        print()
        print("--- Dummy-CEBRA Control ---")
        for cond in ["Tracking", "Playback"]:
            sub = merged[merged["Condition"] == cond]
            print(f"  {cond}: SR_true={sub['SR'].mean():.4f}, "
                  f"SR_dummy={sub['SR_dummy'].mean():.4f}, "
                  f"R2_drive_true={sub['R2_drive'].mean():.6f}, "
                  f"R2_drive_dummy={sub['R2_drive_dummy'].mean():.6f}")
        gate = (merged["R2_drive"] > merged["R2_drive_dummy"]).sum()
        print(f"  R2_drive gate passed: {gate}/{len(merged)} "
              f"session-conditions")

    # Visualization: SR and R2_drive bar plots
    fig, axes = plt.subplots(1, 3, figsize=(9, 3.5))

    # Panel A: SR true vs shuffle vs dummy
    if dummy_df is not None:
        merged = baseline_df.merge(dummy_df,
                                   on=["Subject", "Session_Idx",
                                       "Headstage", "Condition"])
        plot_data = pd.melt(merged,
                            id_vars=["Condition"],
                            value_vars=["SR", "SR_shuffle", "SR_dummy"],
                            var_name="Type", value_name="Skewness Ratio")
    else:
        plot_data = pd.melt(baseline_df,
                            id_vars=["Condition"],
                            value_vars=["SR", "SR_shuffle"],
                            var_name="Type", value_name="Skewness Ratio")
    sns.barplot(data=plot_data, x="Condition", y="Skewness Ratio",
                hue="Type", ax=axes[0],
                palette={"SR": "#440154", "SR_shuffle": "#B2B2B2",
                         "SR_dummy": "#fde725"})
    axes[0].set_title("Skewness Ratio")
    axes[0].legend(fontsize=5)

    # Panel B: R2_drive true vs shuffle
    sns.barplot(data=baseline_df.melt(
        id_vars=["Condition"],
        value_vars=["R2_drive", "R2_drive_shuffle"],
        var_name="Type", value_name="R2_drive"),
        x="Condition", y="R2_drive", hue="Type", ax=axes[1],
        palette={"R2_drive": "#440154", "R2_drive_shuffle": "#B2B2B2"})
    axes[1].set_title("R2_drive (True vs Shuffle)")
    axes[1].legend(fontsize=5)

    # Panel C: Tracking vs Playback per-session scatter
    pivot_plot = baseline_df.pivot_table(
        values="SR", index=["Subject", "Session_Idx", "Headstage"],
        columns="Condition").dropna()
    if len(pivot_plot) > 1:
        axes[2].scatter(pivot_plot["Tracking"], pivot_plot["Playback"],
                        c="#440154", s=20, alpha=0.7)
        lims = [min(pivot_plot.min().min(), 0),
                max(pivot_plot.max().max(), 1)]
        axes[2].plot(lims, lims, '--', c='gray', lw=0.8)
        axes[2].set_xlim(lims); axes[2].set_ylim(lims)
        axes[2].set_xlabel("Tracking SR"); axes[2].set_ylabel("Playback SR")
        axes[2].set_title("Per-Session SR")

    plt.suptitle("Baseline CEBRA-Embedded Lie + Dummy Control",
                 y=1.02, fontsize=9, fontweight="bold")
    plt.tight_layout()
    for fmt in ["pdf", "png"]:
        plt.savefig(os.path.join(LIE_OUTPUT_DIR,
                                 f"Baseline_DummyControl.{fmt}"),
                    dpi=150, bbox_inches="tight")
    plt.show()

else:
    print("Skipped: no baseline data available.")

## 2. End-to-End Lie Dynamics Model (discrete matrix_exp default; ODE experimental)

PyTorch implementation of the joint encoder + Lie generator pipeline.

**Architecture:**
1. **Encoder** (Conv1d, first-layer kernel=11 ≈55ms RF; subsequent layers kernel=1):
   neural window -> latent z (dim D_LATENT) — temporal RF matches CEBRA offset10 (~50ms)
2. **SkewBasis**: D(D-1)/2 fixed orthonormal skew-symmetric basis matrices G_i
3. **ControlNet** (small MLP): multi-dim drive u_t -> weights w_i
   -> J(u_t) = sum_i w_i(u_t) * G_i (skew-symmetric by construction)
4. **Dissipation**: L = -C @ C.T (stable) OR unconstrained (ablation flag)
5. **Transition**: dz/dt = (J(u_t) + L) @ z -> discrete matrix_exp (default) or ODE (**experimental**)

**Key design decisions:**
- **Discrete matrix_exp default** (fast, robust, no torchdiffeq dependency)
- **Dynamics loss variance-normalized**: `mse / var(z_true)` — prevents trivial `||z|| -> 0` solution that InfoNCE's scale-invariance would otherwise allow
- **ODE mode is experimental** (requires torchdiffeq): drive interpolation is piecewise-constant (nearest-neighbor); continuous-time claims not yet realized. Prefer `rk4` over `dopri5` if enabled — adaptive stepper may be unstable at bin boundaries
- **Short-window validation**: held-out R2_drive uses random windows of `VAL_ROLLOUT_LEN` bins (not full-epoch rollout), preventing long-range divergence from contaminating the headline metric
- **CONSTRAINED_L flag**: allows ablation between stable (`-CC^T`) and unconstrained dissipation
- **lambda_dyn warmup**: 0 for first N steps, then linear ramp (stabilizes training)
- **Multi-dim drive (future work)**: ControlNet architecture supports arbitrary `drive_dim`, and `build_drive_vector()` can construct multi-dim drive. However, per-epoch drive alignment with TAU_SHIFT is not yet implemented — `train_one_session()` currently uses only `Velocity_x` labels as drive. Extending to e.g. `["Velocity_x", "Position", "Acc_x"]` requires extracting per-epoch multi-dim vectors aligned to epoch boundaries. Feature ablation (zero out individual dims, measure R2_drive drop) is the planned protocol once multi-dim is implemented.

In [ ]:
# ============================================================
# Phase 1 -- PyTorch Model Components
# ============================================================

# --- Skew-symmetric basis ---

def make_skew_basis(dim):
    """Create orthonormal basis for so(dim): dim*(dim-1)/2 skew-symmetric matrices.

    Each basis matrix G_k has exactly two non-zero entries:
    G_k[i,j] = 1/sqrt(2), G_k[j,i] = -1/sqrt(2).
    Normalized so that ||G_k||_F = 1 and <G_i, G_j> = delta_ij.
    """
    n_basis = dim * (dim - 1) // 2
    basis = torch.zeros(n_basis, dim, dim)
    k = 0
    for i in range(dim):
        for j in range(i + 1, dim):
            G = torch.zeros(dim, dim)
            G[i, j] = 1.0 / (2.0 ** 0.5)
            G[j, i] = -1.0 / (2.0 ** 0.5)
            basis[k] = G
            k += 1
    return basis  # (n_basis, dim, dim)


class SkewBasis(nn.Module):
    """Fixed orthonormal skew-symmetric basis for so(dim).

    Holds dim*(dim-1)/2 orthonormal skew matrices G_i as a fixed buffer.
    The actual coefficients w_i(u_t) come from ControlNet, making
    J(u_t) = sum_i w_i(u_t) * G_i skew-symmetric by construction.

    (No learnable parameters — the ControlNet learns the mapping from drive.)
    """
    def __init__(self, dim):
        super().__init__()
        self.dim = dim
        n_basis = dim * (dim - 1) // 2
        self.register_buffer('basis', make_skew_basis(dim))  # (n_basis, dim, dim)


class ControlNet(nn.Module):
    """Nonlinear control network: multi-dim drive -> basis weights.

    J(u_t) = sum_i w_i(u_t) * G_i, where w_i come from a small MLP.
    """
    def __init__(self, drive_dim, n_basis, hidden=None):
        super().__init__()
        if hidden is None:
            hidden = CONTROL_HIDDEN
        layers = []
        in_dim = drive_dim
        for h in hidden:
            layers.extend([nn.Linear(in_dim, h), nn.ReLU()])
            in_dim = h
        layers.append(nn.Linear(in_dim, n_basis))
        self.net = nn.Sequential(*layers)
        self.n_basis = n_basis

    def forward(self, u_t):
        """u_t: (*, drive_dim) -> w_i: (*, n_basis)."""
        w = self.net(u_t)
        w = torch.tanh(w)  # smooth, bounded [-1, 1]
        return w


class Dissipation(nn.Module):
    """Leak/dissipation matrix L.

    Two modes:
    - CONSTRAINED_L=True:  L = -C @ C.T (guarantees Re(eig) <= 0)
    - CONSTRAINED_L=False: L is unconstrained (dim, dim) matrix
    """
    def __init__(self, dim, constrained=False):
        super().__init__()
        self.dim = dim
        self.constrained = constrained
        if constrained:
            self.C = nn.Parameter(torch.randn(dim, dim) * 0.1)
        else:
            self.L_raw = nn.Parameter(torch.zeros(dim, dim))

    def forward(self):
        """Return L matrix (dim, dim)."""
        if self.constrained:
            return -self.C @ self.C.T
        return self.L_raw

    def get_L_numpy(self):
        """Return L as numpy array."""
        return self.forward().detach().cpu().numpy()


class LieODECell(nn.Module):
    """Lie dynamics transition cell: dz/dt = (J(u_t) + L) @ z.

    Supports two transition modes:
    - discrete: z_{t+1} = matrix_exp(dt * (J(u_t) + L)) @ z_t
    - ode:      z(t) = odeint(dz/dt, z_0, t_span) (requires torchdiffeq)
    """
    def __init__(self, dim, n_basis, constrained_L=False, use_ode=False,
                 ode_method="rk4"):
        super().__init__()
        self.dim = dim
        self.use_ode = use_ode
        self.ode_method = ode_method
        self.skew_basis = SkewBasis(dim)
        self.control = ControlNet(1, n_basis)
        self.dissipation = Dissipation(dim, constrained=constrained_L)

    def set_control_input_dim(self, drive_dim):
        """Rebuild control net if drive_dim changes."""
        if self.control.net[0].in_features != drive_dim:
            n_basis = self.dim * (self.dim - 1) // 2
            self.control = ControlNet(drive_dim, n_basis)

    def compute_generator(self, u_t):
        """Compute J(u_t), L, and full generator A = J + L.

        Args:
            u_t: (*, drive_dim)

        Returns:
            A: (*, dim, dim) full generator
            J_weighted: (*, dim, dim) skew-symmetric component
            L: (dim, dim) dissipation
        """
        L = self.dissipation.forward()      # (dim, dim)
        w = self.control(u_t)               # (*, n_basis)
        G = self.skew_basis.basis           # (n_basis, dim, dim)
        J_weighted = (w[..., None, None] * G).sum(dim=-3)  # (*, dim, dim)
        A = J_weighted + L
        return A, J_weighted, L

    def forward_discrete(self, z_0, drive_seq, dt=0.005):
        """Rollout using discrete matrix_exp.

        z_0: (B, D)
        drive_seq: (B, T, D_drive)
        Returns: z_seq (B, T+1, D) with z_seq[:, 0] = z_0
        """
        B, T, _ = drive_seq.shape
        z_seq = [z_0]
        z_t = z_0
        for t in range(T):
            A, _, _ = self.compute_generator(drive_seq[:, t, :])
            A_dt = A * dt
            exp_A = torch.matrix_exp(A_dt)
            z_t = torch.bmm(exp_A, z_t.unsqueeze(-1)).squeeze(-1)
            z_seq.append(z_t)
        return torch.stack(z_seq, dim=1)

    def _ode_func(self, t, z, drive_interp):
        """ODE RHS: dz/dt = (J(u_t) + L) @ z."""
        u_t = drive_interp(t)
        A, _, _ = self.compute_generator(u_t)
        return torch.bmm(A, z.unsqueeze(-1)).squeeze(-1)

    def forward_ode(self, z_0, drive_seq, dt=0.005):
        """Rollout using ODE solver (requires torchdiffeq).

        z_0: (B, D)
        drive_seq: (B, T, D_drive)
        Returns: z_seq (B, T+1, D)
        """
        B, T, _ = drive_seq.shape
        t_span = torch.linspace(0, T * dt, T + 1, device=z_0.device)

        def drive_interp(t_val):
            idx = (t_val / dt).long().clamp(0, T - 1)
            return drive_seq[torch.arange(B, device=t_val.device), idx, :]

        z_seq = torchdiffeq.odeint(
            lambda t, z: self._ode_func(t, z, drive_interp),
            z_0, t_span, method=self.ode_method,
            options={'step_size': dt} if self.ode_method == 'rk4' else {}
        )
        return z_seq.permute(1, 0, 2)

    def forward(self, z_0, drive_seq, dt=0.005):
        """Rollout: z_0 -> z_seq."""
        if self.use_ode and HAS_TORCHDIFFEQ:
            if self.ode_method == "dopri5":
                print("WARNING: USE_ODE is experimental; drive is piecewise-constant "
                      "(nearest-neighbor interp). Adaptive dopri5 may be unstable "
                      "at bin boundaries — prefer rk4 or discrete matrix_exp.")
            return self.forward_ode(z_0, drive_seq, dt)
        return self.forward_discrete(z_0, drive_seq, dt)


class Encoder(nn.Module):
    """Temporal conv encoder: offset-window neural -> latent z (preserves T).

    Only the first Conv1d layer uses kernel=offset (default 11 bins = 55ms at
    dt=0.005).  All subsequent layers use kernel=1, so the temporal receptive
    field is exactly `offset` bins — comparable to CEBRA offset10 (~50ms).

    Input:  (B, T, N) or (T, N)
    Output: (B, T, D_LATENT) or (T, D_LATENT)
    """
    def __init__(self, n_neurons, latent_dim, hidden=None, offset=11):
        super().__init__()
        if hidden is None:
            hidden = ENCODER_HIDDEN
        if offset % 2 == 0:
            raise ValueError(f"offset must be odd to preserve T; got {offset}")
        pad = offset // 2   # symmetric padding preserves length
        in_c = n_neurons
        layers = []
        # First layer: temporal context via offset-kernel Conv1d
        layers += [nn.Conv1d(in_c, hidden[0], offset, padding=pad), nn.ReLU()]
        in_c = hidden[0]
        # Subsequent layers: kernel=1 (pointwise, no additional temporal RF)
        for h in hidden[1:]:
            layers += [nn.Conv1d(in_c, h, 1), nn.ReLU()]
            in_c = h
        layers += [nn.Conv1d(in_c, latent_dim, 1)]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        """x: (B,T,N) or (T,N) -> (B,T,D) or (T,D)."""
        single = (x.dim() == 2)
        if single:
            x = x.unsqueeze(0)   # (1, T, N)
        # Conv1d expects (B, C, T)
        z = self.net(x.transpose(1, 2)).transpose(1, 2)  # (B, T, D)
        return z.squeeze(0) if single else z


class SkieurLieODE(nn.Module):
    """Full end-to-end model: Encoder + LieODECell.

    forward modes:
    - "encode":  neural -> latent z only
    - "rollout": z_0 + drive_seq -> predicted trajectory
    - "full":    neural -> z -> rollout -> (z_true, z_pred)
    """
    def __init__(self, n_neurons, latent_dim, drive_dim,
                 constrained_L=False, use_ode=False, ode_method="rk4"):
        super().__init__()
        n_basis = latent_dim * (latent_dim - 1) // 2
        self.latent_dim = latent_dim
        self.encoder = Encoder(n_neurons, latent_dim)
        self.lie_cell = LieODECell(latent_dim, n_basis,
                                   constrained_L=constrained_L,
                                   use_ode=use_ode, ode_method=ode_method)
        self.lie_cell.set_control_input_dim(drive_dim)

    def encode(self, neural):
        """neural: (T, N) or (B, T, N) -> z: (T, D) or (B, T, D)."""
        return self.encoder(neural)

    def rollout(self, z_0, drive_seq, dt=0.005):
        """z_0: (B, D), drive_seq: (B, T, D_drive) -> z_seq: (B, T+1, D)."""
        return self.lie_cell(z_0, drive_seq, dt)

    def forward(self, neural_seq, drive_seq, dt=0.005):
        """Full forward pass.

        neural_seq: (B, T, N)
        drive_seq:  (B, T, D_drive)
        Returns: z_true (B, T, D), z_pred (B, T+1, D)
        """
        # Conv encoder preserves T — no reshape/flatten needed
        z_true = self.encode(neural_seq)  # (B, T, D)
        z_pred = self.rollout(z_true[:, 0, :], drive_seq, dt)
        return z_true, z_pred

    def get_generator_matrices(self, n_drive_samples=100):
        """Return metrics from the drive-dependent generator J(u) = sum w_i(u) G_i.

        CRITICAL: a successful model learns w_i(u) ≈ -w_i(-u) (rotation direction
        follows velocity sign).  With symmetric drive distribution N(0,1), the
        MATRIX average E_u[J(u)] ≈ 0 cancels out — exactly when rotation is
        strongest.  We therefore compute metrics PER SAMPLE and average:

          SR      = mean_u [ ||J(u)|| / (||J(u)|| + ||L||) ]
          |Real|  = mean_u [ |Real(eig(J(u) + L))| ]
          |Imag|  = mean_u [ |Imag(eig(J(u) + L))| ]

        Returns:
            J_avg:  (D, D) mean matrix (may be near zero — do NOT use for SR/eig)
            L:      (D, D) dissipation matrix
            sr_mean:     float, per-sample-averaged skewness ratio
            eig_real:    float, per-sample-averaged |Real(eigenvalue)|
            eig_imag:    float, per-sample-averaged |Imag(eigenvalue)|
        """
        with torch.no_grad():
            dev = next(self.parameters()).device
            d_drive = self.lie_cell.control.net[0].in_features
            u_samples = torch.randn(n_drive_samples, d_drive, device=dev)
            _, J_samples, L = self.lie_cell.compute_generator(u_samples)
            L_np = L.cpu().numpy()
            L_fro = np.linalg.norm(L_np)

            # Per-sample metrics
            J_np = J_samples.cpu().numpy()  # (n_samples, D, D)
            sr_vals = []
            eig_real_vals, eig_imag_vals = [], []
            for k in range(n_drive_samples):
                J_k = J_np[k]
                j_norm = np.linalg.norm(J_k)
                sr_vals.append(j_norm / (j_norm + L_fro + 1e-9))
                eigvals = np.linalg.eigvals(J_k + L_np)
                eig_real_vals.append(np.mean(np.abs(np.real(eigvals))))
                eig_imag_vals.append(np.mean(np.abs(np.imag(eigvals))))

            sr_mean = float(np.mean(sr_vals))
            eig_real = float(np.mean(eig_real_vals))
            eig_imag = float(np.mean(eig_imag_vals))

            # Average matrix (for reference only — may cancel for odd w(u))
            J_avg = J_samples.mean(dim=0).cpu().numpy()

        return J_avg, L_np, sr_mean, eig_real, eig_imag


# --- Quick sanity test ---
n_test = n_data_all[0].shape[0]
d_drive = len(DRIVE_KEYS)
model = SkieurLieODE(n_test, D_LATENT, d_drive,
                     constrained_L=CONSTRAINED_L,
                     use_ode=USE_ODE, ode_method=ODE_METHOD)
model.to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f"Model: {n_params:,} parameters")
print(f"  Encoder: Conv1d offset-11, {n_test} -> {D_LATENT} (comparable to CEBRA offset10)")
print(f"  Skew basis: {D_LATENT}*(D_LATENT-1)/2 = {D_LATENT*(D_LATENT-1)//2}")
print(f"  Drive dim: {d_drive}")
print(f"  CONSTRAINED_L: {CONSTRAINED_L}")
print(f"  USE_ODE: {USE_ODE}")

# Quick smoke test
B, T = 8, MINI_TRAJ_LEN
x_test = torch.randn(B, T, n_test, device=DEVICE)
d_test = torch.randn(B, T, d_drive, device=DEVICE)
z_true, z_pred = model(x_test, d_test)
print(f"  Smoke test: z_true {tuple(z_true.shape)}, z_pred {tuple(z_pred.shape)}")
# Check skewness (per-sample average, NOT matrix-average)
J_avg, L_m, sr, eig_r, eig_i = model.get_generator_matrices()
print(f"  Initial SR (per-sample avg): {sr:.4f}, |Real|: {eig_r:.4f}, |Imag|: {eig_i:.4f}")
print(f"  (J_avg norm: {np.linalg.norm(J_avg):.4f} — may be near-zero for odd w(u); SR/eig use per-sample avg)")
print("Model instantiation OK.")

In [ ]:
# ============================================================
# Phase 2 -- Joint Training Loop (per session)
# ============================================================
# Loss = InfoNCE(z; behavior labels) + lambda_dyn * Dynamics_MSE(z_pred, z_true)
#
# Addressing 3 core challenges:
#   1. Batching: mini-trajectories (T=MINI_TRAJ_LEN) -> InfoNCE on
#      ALL flattened frames; Dynamics on per-trajectory rollout.
#   2. lambda balancing: warmup (lambda=0 for N steps) then linear ramp.
#   3. Dissipation constraint: ablation flag CONSTRAINED_L.


class TrajectoryDataset(Dataset):
    """Dataset of mini-trajectories for joint training."""

    def __init__(self, traj_n, traj_l, traj_d):
        self.traj_n = torch.tensor(traj_n, dtype=torch.float32)
        self.traj_l = torch.tensor(traj_l, dtype=torch.float32)
        self.traj_d = torch.tensor(traj_d, dtype=torch.float32)

    def __len__(self):
        return len(self.traj_n)

    def __getitem__(self, idx):
        return self.traj_n[idx], self.traj_l[idx], self.traj_d[idx]


def get_lambda_dyn(step, warmup, lambda_max):
    """Compute effective lambda_dyn with warmup.

    0 for step < warmup, then linear ramp to lambda_max over warmup steps.
    """
    if step < warmup:
        return 0.0
    if step < 2 * warmup:
        return lambda_max * (step - warmup) / warmup
    return lambda_max


def compute_r2_drive_rollout(model, ep_n, ep_drive, dt=0.005,
                              window_len=None, n_windows=20):
    """Cross-validated R2_drive using SHORT-WINDOW rollout.

    CRITICAL: Full-epoch rollout with unconstrained L diverges exponentially
    over hundreds of timesteps, contaminating R2_drive with numerical instability
    rather than measuring learned dynamics quality.

    Instead, we sample n_windows short windows (each of length window_len,
    matching the training mini-trajectory length) and average R2_drive across
    them. This isolates the model's ability to predict short-term dynamics
    — exactly what the training objective optimizes.

    IMPORTANT — train/val consistency: each window is encoded independently
    from raw neural data (not sliced from a full-epoch encoding).  This
    matches the training pipeline (create_mini_trajectories then encode),
    ensuring identical edge-padding semantics and eliminating a
    distribution shift between training and validation encodings.

    Args:
        model: trained SkieurLieODE
        ep_n: (T_epoch, N) neural data for one epoch
        ep_drive: (T_epoch, D_drive) drive for one epoch
        dt: bin size
        window_len: rollout window length (default: VAL_ROLLOUT_LEN)
        n_windows: number of short windows to sample

    Returns:
        r2_drive: float (averaged across windows)
    """
    if window_len is None:
        window_len = VAL_ROLLOUT_LENS[0]  # shortest scale by default

    model.eval()
    with torch.no_grad():
        n_t = torch.tensor(ep_n, dtype=torch.float32, device=DEVICE)
        d_t = torch.tensor(ep_drive, dtype=torch.float32, device=DEVICE)
        T_epoch = n_t.shape[0]

        # Precompute leak transition (L is constant, same for all windows)
        L_mat = model.lie_cell.dissipation.forward()       # (D, D)
        exp_L_dt = torch.matrix_exp(L_mat * dt)            # (D, D) — cached once

        if T_epoch < window_len + 1:
            starts = [0]
            n_actual = 1
        else:
            n_actual = min(n_windows, T_epoch - window_len)
            starts = np.random.choice(T_epoch - window_len, size=n_actual,
                                      replace=False)

        mse_full_wins, mse_leak_wins = [], []
        for s in starts:
            end = s + window_len

            # --- Per-window encoding (matches training pipeline) ---
            # Slice RAW neural, then encode — same edge-padding semantics
            # as create_mini_trajectories -> model(batch_n).
            n_win = n_t[s:end]                 # (W, N) raw
            d_win = d_t[s:end]                 # (W, D_drive)
            z_win = model.encode(n_win)        # (W, D) per-window encoding

            # Full rollout: z_0 -> z_1_pred ... z_{W-1}_pred
            z_pred_full = model.rollout(
                z_win[0:1], d_win.unsqueeze(0), dt).squeeze(0)  # (W+1, D)

            # Leak-only rollout (uses cached exp_L_dt)
            z_leak = [z_win[0:1]]
            z_t = z_win[0:1]
            for _ in range(len(d_win)):
                z_t = torch.mm(exp_L_dt, z_t.T).T
                z_leak.append(z_t)
            z_pred_leak = torch.cat(z_leak, dim=0)  # (W+1, D)

            # MSE: align z_pred[1:-1] vs z_win[1:] -> both (W-1, D)
            mse_full_wins.append(
                F.mse_loss(z_pred_full[1:-1], z_win[1:]).item())
            mse_leak_wins.append(
                F.mse_loss(z_pred_leak[1:-1], z_win[1:]).item())

        if not mse_full_wins:
            return float('nan')

        mse_full_avg = np.mean(mse_full_wins)
        mse_leak_avg = np.mean(mse_leak_wins)
        r2_drive = 1.0 - (mse_full_avg / (mse_leak_avg + 1e-9))
        return r2_drive


def compute_r2_drive_shuffle(model, ep_n, ep_drive, dt=0.005,
                              window_len=None, n_windows=5, n_shuffles=10):
    """E2E null control: circular-shift drive on FIXED trained encoder.

    Uses np.roll (circular shift) rather than np.random.permutation.
    This PRESERVES the autocorrelation structure and power spectrum of the
    drive signal x(t) (which is slowly-varying and highly autocorrelated),
    destroying only the temporal ALIGNMENT with the neural state.  This is
    the correct null: same distribution, same autocorrelation, random timing.

    (cf. lie_algebra_method_description.md section 12.3 — permutation destroys
    both alignment AND spectrum, confounding the null hypothesis.)

    n_windows is reduced to 5 (vs 20 for the true R2_drive) to keep
    validation time manageable: each shuffle is a full per-window
    encode + rollout, and n_shuffles=10 × n_windows multiplies cost.

    Returns:
        r2_drive_shuffle: float (mean across circular-shift realizations)
    """
    if window_len is None:
        window_len = VAL_ROLLOUT_LENS[0]  # shortest scale by default

    T_drive = len(ep_drive)
    if T_drive < 2:
        return float('nan')

    # Minimum shift >= MINI_TRAJ_LEN: shifts smaller than the trajectory
    # length leave the shuffled drive highly correlated with the original
    # (anti-conservative null).  We enforce a gap at least as large as
    # the rollout window so the null genuinely breaks alignment.
    min_shift = max(1, window_len)
    if T_drive <= 2 * min_shift:
        min_shift = max(1, T_drive // 3)

    vals = []
    for _ in range(n_shuffles):
        shift = np.random.randint(min_shift, T_drive - min_shift)
        ep_drive_shuf = np.roll(ep_drive, shift, axis=0)
        r2d = compute_r2_drive_rollout(model, ep_n, ep_drive_shuf,
                                        dt=dt, window_len=window_len,
                                        n_windows=n_windows)
        if not np.isnan(r2d):
            vals.append(r2d)
    return np.mean(vals) if vals else float('nan')


def train_one_session(model, n_data_session, f_df, session_idx,
                       n_epochs_train=N_EPOCHS_TRAIN,
                       lambda_dyn=LAMBDA_DYN,
                       lambda_warmup=LAMBDA_DYN_WARMUP):
    """Train the end-to-end model on one session.

    Returns:
        model: trained model
        history: dict with loss/lambda curves
        val_metrics: dict with per-condition held-out metrics
    """
    model.train()
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR,
                                  weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=n_epochs_train, eta_min=1e-5)

    # Extract epochs for both conditions
    cond_data = {}
    # Collect all labels first for per-session standardization
    all_labels_session = []
    for val, label in [(0.0, "Tracking"), (1.0, "Playback")]:
        epochs_n, epochs_l, _ = extract_epochs(
            n_data_session, f_df, val, dt, label_col="Velocity_x")
        all_labels_session.extend(epochs_l)

    # Standardize labels per session (they ARE the drive when DRIVE_KEYS=["Velocity_x"])
    if all_labels_session:
        lab_cat = np.concatenate(all_labels_session)
        mu_l, std_l = np.mean(lab_cat), np.std(lab_cat)
    else:
        mu_l, std_l = 0.0, 1.0
    if std_l < 1e-9:
        std_l = 1.0

    for val, label in [(0.0, "Tracking"), (1.0, "Playback")]:
        epochs_n, epochs_l, _ = extract_epochs(
            n_data_session, f_df, val, dt, label_col="Velocity_x")
        # Use standardized labels as drive (correctly aligned + TAU_SHIFT-trimmed)
        # Reshape to (T, 1) for single-drive ControlNet input
        drive_epochs = [((el - mu_l) / std_l).reshape(-1, 1).astype(np.float32)
                        for el in epochs_l]
        valid_idx = [i for i in range(len(epochs_n))
                     if epochs_n[i].shape[0] >= MINI_TRAJ_LEN]
        cond_data[label] = {
            'n': [epochs_n[i] for i in valid_idx],
            'l': [epochs_l[i] for i in valid_idx],
            'd': [drive_epochs[i] for i in valid_idx]
        }

    # Multi-dim drive not yet implemented in the aligned epoch path.
    # (Would require per-epoch multi-dim drive extraction with TAU_SHIFT alignment.)
    if len(DRIVE_KEYS) > 1 or DRIVE_KEYS != ["Velocity_x"]:
        raise NotImplementedError(
            f"Multi-dim drive not yet implemented. DRIVE_KEYS={DRIVE_KEYS}. "
            f"Set DRIVE_KEYS=['Velocity_x'] for now. "
            f"To implement: extract per-epoch multi-dim drive vectors aligned "
            f"to epoch boundaries with TAU_SHIFT, matching epochs_l alignment.")

    # Train/val split on epochs (requires >=2 epochs per condition for valid split)
    all_train_n, all_train_l, all_train_d = [], [], []
    val_epochs = {}
    n_skipped_val = 0
    for cond_name in ["Tracking", "Playback"]:
        cd = cond_data[cond_name]
        n_ep = len(cd['n'])
        if n_ep < 2:
            print(f"  Session {session_idx} {cond_name}: only {n_ep} epoch(s), "
                  f"skipping (need >=2 for train/val split)")
            n_skipped_val += 1
            val_epochs[cond_name] = {'n': [], 'l': [], 'd': []}
            continue
        n_train = max(1, int(n_ep * TRAIN_VAL_SPLIT))
        # Training epochs
        for i in range(n_train):
            all_train_n.append(cd['n'][i])
            all_train_l.append(cd['l'][i])
            all_train_d.append(cd['d'][i])
        # Held-out validation epochs
        val_epochs[cond_name] = {
            'n': cd['n'][n_train:],
            'l': cd['l'][n_train:],
            'd': cd['d'][n_train:],
        }

    if n_skipped_val >= 2 or not all_train_n:
        print(f"  Session {session_idx}: insufficient epochs for train/val, skipping")
        return model, {}, {}

    history = {'loss_total': [], 'loss_infonce': [], 'loss_dyn': [],
               'lambda_dyn': [], 'grad_norm': []}
    d_drive = len(DRIVE_KEYS)
    global_step = 0

    for epoch in trange(n_epochs_train, desc=f"Train S{session_idx}",
                        leave=False):
        # Sample mini-trajectories
        traj_n, traj_l, traj_d = create_mini_trajectories(
            all_train_n, all_train_l, all_train_d,
            traj_len=MINI_TRAJ_LEN, n_samples_per_epoch=30)

        if traj_n is None:
            continue

        dataset = TrajectoryDataset(traj_n, traj_l, traj_d)
        dataloader = DataLoader(dataset, batch_size=BATCH_SIZE // MINI_TRAJ_LEN,
                                shuffle=True, drop_last=True)

        epoch_loss_total, epoch_loss_info, epoch_loss_dyn = 0.0, 0.0, 0.0
        n_batches = 0

        for batch_n, batch_l, batch_d in dataloader:
            batch_n = batch_n.to(DEVICE)  # (B, T, N)
            batch_l = batch_l.to(DEVICE)  # (B, T)
            batch_d = batch_d.to(DEVICE)  # (B, T, D_drive)

            B, T_len, N_neurons = batch_n.shape

            # ---- Forward pass ----
            z_true, z_pred = model(batch_n, batch_d, dt)  # (B,T,D), (B,T+1,D)

            # ---- InfoNCE loss (flatten ALL frames across batch) ----
            z_flat = z_true.reshape(-1, D_LATENT)  # (B*T, D)
            l_flat = batch_l.reshape(-1)           # (B*T,)
            loss_info = info_nce_loss(z_flat, l_flat, temperature=TEMPERATURE)

            # ---- Dynamics MSE (trajectory rollout prediction) ----
            # z_pred: (B, T+1, D) = [z_0_pred, z_1_pred, ..., z_T_pred]
            # z_true: (B, T,   D) = [z_0_true, z_1_true, ..., z_{T-1}_true]
            # Align: z_pred[1:-1] (predicted z_1..z_{T-1}) vs z_true[1:] (true z_1..z_{T-1})
            # Both: (B, T-1, D)
            #
            # CRITICAL: InfoNCE is scale-invariant (F.normalize), so ||z|| has no
            # InfoNCE gradient.  Raw MSE ∝ ||z||^2, so the optimizer can trivially
            # shrink ||z|| -> 0 to reduce dynamics loss without learning any real
            # dynamics.  Normalising by variance removes this degenerate path:
            # both numerator and denominator scale with ||z||^2 -> ratio is
            # scale-invariant, matching the R2_drive evaluation philosophy.
            z_tgt = z_true[:, 1:]                       # (B, T-1, D)
            z_prd = z_pred[:, 1:-1]                     # (B, T-1, D)
            mse_dyn = F.mse_loss(z_prd, z_tgt)
            var_z   = z_tgt.var().clamp_min(1e-6)       # per-element variance
            loss_dyn = mse_dyn / var_z

            # ---- Lambda schedule (warmup) ----
            lam = get_lambda_dyn(global_step, lambda_warmup, lambda_dyn)

            # ---- Total loss ----
            loss = loss_info + lam * loss_dyn

            # ---- Optimization ----
            optimizer.zero_grad()
            loss.backward()
            grad_norm = torch.nn.utils.clip_grad_norm_(
                model.parameters(), GRAD_CLIP)
            optimizer.step()

            epoch_loss_total += loss.item()
            epoch_loss_info += loss_info.item()
            epoch_loss_dyn += loss_dyn.item()
            n_batches += 1
            global_step += 1

        if n_batches > 0:
            history['loss_total'].append(epoch_loss_total / n_batches)
            history['loss_infonce'].append(epoch_loss_info / n_batches)
            history['loss_dyn'].append(epoch_loss_dyn / n_batches)
            history['lambda_dyn'].append(lam)
            history['grad_norm'].append(grad_norm.item() if hasattr(grad_norm, 'item') else grad_norm)

        scheduler.step()

    # ---- Multi-scale validation on held-out epochs ----
    val_metrics = {}
    for cond_name in ["Tracking", "Playback"]:
        ve = val_epochs[cond_name]
        r2d_multiscale = {w: [] for w in VAL_ROLLOUT_LENS}
        r2d_sh_multiscale = {w: [] for w in VAL_ROLLOUT_LENS}
        for ep_n, ep_d in zip(ve['n'], ve['d']):
            for wlen in VAL_ROLLOUT_LENS:
                r2d = compute_r2_drive_rollout(model, ep_n, ep_d, dt,
                                                window_len=wlen)
                r2d_multiscale[wlen].append(r2d)
                # Shuffle null (only at shortest scale for efficiency)
                if wlen == VAL_ROLLOUT_LENS[0]:
                    r2d_sh = compute_r2_drive_shuffle(
                        model, ep_n, ep_d, dt, window_len=wlen,
                        n_shuffles=N_SHUFFLES)
                    r2d_sh_multiscale[wlen].append(r2d_sh)
                else:
                    r2d_sh_multiscale[wlen].append(float('nan'))

        val_metrics[cond_name] = {
            'R2_drive_multiscale': {w: np.mean(r2d_multiscale[w])
                                     if r2d_multiscale[w] else float('nan')
                                     for w in VAL_ROLLOUT_LENS},
            'R2_drive_shuffle_multiscale': {w: np.mean(r2d_sh_multiscale[w])
                                             if r2d_sh_multiscale[w] else float('nan')
                                             for w in VAL_ROLLOUT_LENS},
            # Backward-compat: primary metric at shortest scale
            'R2_drive_rollout': np.mean(r2d_multiscale[VAL_ROLLOUT_LENS[0]])
            if r2d_multiscale[VAL_ROLLOUT_LENS[0]] else float('nan'),
            'R2_drive_shuffle': np.mean(r2d_sh_multiscale[VAL_ROLLOUT_LENS[0]])
            if r2d_sh_multiscale[VAL_ROLLOUT_LENS[0]] else float('nan'),
            'n_val_epochs': len(ve['n']),
        }

    return model, history, val_metrics


print("Training utilities ready. Ready to train per session.")
print(f"Training config: N_EPOCHS={N_EPOCHS_TRAIN}, LR={LR}, "
      f"BATCH={BATCH_SIZE}, MINI_TRAJ_LEN={MINI_TRAJ_LEN}")
print(f"lambda_dyn={LAMBDA_DYN}, warmup={LAMBDA_DYN_WARMUP} steps")

In [ ]:
# ============================================================
# Phase 2b -- Run Joint Training (all sessions, multi-seed)
# ============================================================
# Trains one model per session × seed, with multi-scale validation.
# Optionally runs lambda_dyn=0 ablation for comparison.

e2e_results = []       # per session-condition-seed (R2_drive at multiple scales)
e2e_session = []       # per session-seed (SR, eig)
e2e_histories = []     # per session-seed training curves

# Determine training sessions
n_sessions_available = len(n_data_all)
if N_TRAIN_SESSIONS is None:
    n_train = n_sessions_available
else:
    n_train = min(N_TRAIN_SESSIONS, n_sessions_available)
train_indices = np.random.choice(n_sessions_available, size=n_train, replace=False)
train_indices = sorted(train_indices)
print(f"Training on {n_train}/{n_sessions_available} sessions "
      f"(indices: {train_indices.tolist()})")
print(f"Seeds per session: {N_SEEDS}")
print(f"Validation scales: {VAL_ROLLOUT_LENS} bins")

# ---- Optional lambda_dyn=0 ablation (pure InfoNCE, no dynamics constraint) ----
DO_ABLATION = True  # Set False to skip the ablation pass
ablation_results = []  # per session SR without dynamics loss

for idx in tqdm(train_indices, desc="Joint Training"):
    n_data_session = n_data_all[idx]
    f_df = f_data_all[idx]
    hs_label = "hs0" if idx < n_hs0 else "hs1"
    n_neurons = n_data_session.shape[0]
    d_drive = len(DRIVE_KEYS)

    # ---- Pre-compute pooled TR+PB standardization (matching training) ----
    all_v = np.concatenate([
        f_df[f_df["Condition"] == val]["Velocity_x"].values
        for val in [0.0, 1.0]])
    mu_pool, std_pool = np.mean(all_v), np.std(all_v)
    if std_pool < 1e-9: std_pool = 1.0

    # ---- Multi-scale R2_drive accumulator (averaged across seeds) ----
    seed_r2d = {cond: {s: [] for s in VAL_ROLLOUT_LENS}
                for cond in ["Tracking", "Playback"]}
    seed_r2d_sh = {cond: {s: [] for s in VAL_ROLLOUT_LENS}
                   for cond in ["Tracking", "Playback"]}
    seed_sr, seed_eig_r, seed_eig_i = [], [], []
    seed_sr_tr, seed_sr_pb = [], []  # per-condition SR
    seed_losses = []

    for seed in range(N_SEEDS):
        # Set per-seed randomness
        torch.manual_seed(RANDOM_SEED + idx * 100 + seed)
        np.random.seed(RANDOM_SEED + idx * 100 + seed)

        model = SkieurLieODE(n_neurons, D_LATENT, d_drive,
                             constrained_L=CONSTRAINED_L,
                             use_ode=USE_ODE, ode_method=ODE_METHOD)
        model.to(DEVICE)

        model, history, val_metrics = train_one_session(
            model, n_data_session, f_df, idx)

        e2e_histories.append(history)
        J_avg, L_mat, sr, re, im = model.get_generator_matrices()
        seed_sr.append(sr)
        seed_eig_r.append(re)
        seed_eig_i.append(im)
        seed_losses.append(history['loss_total'][-1]
                           if history['loss_total'] else float('nan'))

        for cond_name in ["Tracking", "Playback"]:
            vm = val_metrics.get(cond_name, {})
            # Multi-scale R2_drive (keyed by window length)
            r2d_dict = vm.get('R2_drive_multiscale', {})
            r2d_sh_dict = vm.get('R2_drive_shuffle_multiscale', {})
            for wlen in VAL_ROLLOUT_LENS:
                seed_r2d[cond_name][wlen].append(
                    r2d_dict.get(wlen, float('nan')))
                seed_r2d_sh[cond_name][wlen].append(
                    r2d_sh_dict.get(wlen, float('nan')))

        # ---- Per-condition SR (pooled standardization, accumulated across seeds) ----
        with torch.no_grad():
            L_np = model.lie_cell.dissipation.get_L_numpy()
            L_fro = np.linalg.norm(L_np)
            for val, cond_name in [(0.0, "Tracking"), (1.0, "Playback")]:
                v_cond = f_df[f_df["Condition"] == val]["Velocity_x"].values
                if len(v_cond) > 100:
                    v_std = (v_cond - mu_pool) / std_pool  # pooled standardization
                    u_samples = torch.tensor(
                        v_std[:1000].reshape(-1, 1),
                        dtype=torch.float32, device=DEVICE)
                    _, J_samples, _ = model.lie_cell.compute_generator(u_samples)
                    J_np = J_samples.cpu().numpy()
                    sr_vals = [np.linalg.norm(J_np[k]) /
                               (np.linalg.norm(J_np[k]) + L_fro + 1e-9)
                               for k in range(len(J_np))]
                    (seed_sr_tr if cond_name == "Tracking" else seed_sr_pb).append(
                        float(np.mean(sr_vals)))
                else:
                    (seed_sr_tr if cond_name == "Tracking" else seed_sr_pb).append(
                        float('nan'))

        del model; gc.collect(); torch.cuda.empty_cache()

    # ---- Aggregate across seeds ----
    e2e_session.append({
        "Subject": "SKIEUR", "Session_Idx": idx, "Headstage": hs_label,
        "Space": "E2E_LieDynamics", "D_LATENT": D_LATENT,
        "SR": np.mean(seed_sr), "SR_sem": np.std(seed_sr) / max(1, N_SEEDS)**0.5,
        "SR_Tracking": np.mean(seed_sr_tr) if seed_sr_tr else float('nan'),
        "SR_Playback": np.mean(seed_sr_pb) if seed_sr_pb else float('nan'),
        "Eig_Real_Mean": np.mean(seed_eig_r), "Eig_Imag_Mean": np.mean(seed_eig_i),
        "Loss_final": np.mean(seed_losses),
        "N_Seeds": N_SEEDS, "N_Neurons": n_neurons,
    })

    for cond_name in ["Tracking", "Playback"]:
        for wlen in VAL_ROLLOUT_LENS:
            r2_vals = seed_r2d[cond_name][wlen]
            r2_sh_vals = seed_r2d_sh[cond_name][wlen]
            e2e_results.append({
                "Subject": "SKIEUR", "Session_Idx": idx,
                "Headstage": hs_label, "Condition": cond_name,
                "Window_Bins": wlen,
                "R2_drive_rollout": np.mean(r2_vals) if r2_vals else float('nan'),
                "R2_drive_rollout_sem": np.std(r2_vals) / max(1, len(r2_vals))**0.5
                if r2_vals else float('nan'),
                "R2_drive_shuffle": np.mean(r2_sh_vals) if r2_sh_vals else float('nan'),
            })

    # ---- lambda_dyn=0 ablation: post-hoc OLS Lie on frozen embedding ----
    # The encoder from λ=0 training has NOT seen the dynamics constraint.
    # Post-hoc OLS Lie fit asks: does InfoNCE alone produce rotational
    # embedding structure, without the Lie loss?  If SR_ols_abl ≈ SR_e2e,
    # the rotation comes from InfoNCE, not from the dynamics constraint.
    if DO_ABLATION:
        torch.manual_seed(RANDOM_SEED + idx * 100)
        np.random.seed(RANDOM_SEED + idx * 100)
        model_abl = SkieurLieODE(n_neurons, D_LATENT, d_drive,
                                 constrained_L=CONSTRAINED_L,
                                 use_ode=USE_ODE, ode_method=ODE_METHOD)
        model_abl.to(DEVICE)
        model_abl, hist_abl, _ = train_one_session(
            model_abl, n_data_session, f_df, idx,
            lambda_dyn=0.0, lambda_warmup=0)

        # Post-hoc OLS Lie fit on frozen λ=0 embedding (per-condition)
        for val, cond_name in [(0.0, "Tracking"), (1.0, "Playback")]:
            epochs_n, epochs_l, _ = extract_epochs(
                n_data_session, f_df, val, dt, label_col="Velocity_x")
            if not epochs_n:
                continue
            # Pool condition epochs, encode, OLS fit
            ep_cat = np.concatenate(epochs_n, axis=0)
            lab_cat = np.concatenate(epochs_l, axis=0)
            with torch.no_grad():
                emb_cat = model_abl.encode(
                    torch.tensor(ep_cat, dtype=torch.float32, device=DEVICE)
                ).cpu().numpy()
            J_s, sr_ols, r2, J_ols, r2d = fit_lie_algebra_with_leak(emb_cat, lab_cat)
            re_ols, im_ols = compute_eigenvalue_metrics(J_ols)
            ablation_results.append({
                "Session_Idx": idx, "Headstage": hs_label,
                "Condition": cond_name,
                "SR_lambda0_ols": sr_ols,
                "R2_drive_lambda0_ols": r2d,
                "Eig_Real_lambda0": re_ols,
                "Eig_Imag_lambda0": im_ols,
            })
        del model_abl; gc.collect(); torch.cuda.empty_cache()

e2e_df = pd.DataFrame(e2e_results)
e2e_session_df = pd.DataFrame(e2e_session)
ablation_df = pd.DataFrame(ablation_results) if ablation_results else None

print(f"Trained {n_train} sessions × {N_SEEDS} seeds = "
      f"{n_train * N_SEEDS} models.")
print()

# --- Multi-scale R2_drive summary ---
print("--- Multi-scale R2_drive (mean across sessions × seeds) ---")
r2_pivot = e2e_df.pivot_table(
    values="R2_drive_rollout", index="Window_Bins", columns="Condition",
    aggfunc="mean").round(6)
print(r2_pivot.to_string())

# --- SR summary ---
print("\n--- Skewness Ratio ---")
print(f"  SR (N(0,1) diagnostic): {e2e_session_df['SR'].mean():.4f} "
      f"± {e2e_session_df['SR'].sem():.4f}  [from get_generator_matrices, random-drive]")
if 'SR_Tracking' in e2e_session_df.columns:
    print(f"  SR_Tracking (empirical):  {e2e_session_df['SR_Tracking'].mean():.4f} "
          f"± {e2e_session_df['SR_Tracking'].sem():.4f}  [condition-specific drive]")
    print(f"  SR_Playback (empirical):  {e2e_session_df['SR_Playback'].mean():.4f} "
          f"± {e2e_session_df['SR_Playback'].sem():.4f}  [condition-specific drive]")
    common_sr = e2e_session_df[['SR_Tracking', 'SR_Playback']].dropna()
    if len(common_sr) > 1:
        t_sr, p_sr = ttest_rel(common_sr['SR_Tracking'], common_sr['SR_Playback'])
        print(f"  SR TR vs PB paired t-test: t={t_sr:.3f}, p={p_sr:.4f}")

# --- lambda_dyn=0 ablation summary (post-hoc OLS on frozen embedding) ---
if ablation_df is not None:
    print("\n--- lambda_dyn=0 Ablation (post-hoc OLS Lie on frozen embedding) ---")
    # Compare E2E R2_drive vs OLS R2_drive on λ=0 embedding (shortest window)
    e2e_short = e2e_df[e2e_df["Window_Bins"] == VAL_ROLLOUT_LENS[0]]
    merged_abl = e2e_short.merge(ablation_df, on=["Session_Idx", "Condition"])
    for cond in ["Tracking", "Playback"]:
        sub = merged_abl[merged_abl["Condition"] == cond]
        sr_col = f"SR_{cond}"
        sr_e2e_cond = e2e_session_df[sr_col].mean() if sr_col in e2e_session_df.columns else float('nan')
        print(f"  {cond}: R2_drive_E2E={sub['R2_drive_rollout'].mean():.6f}, "
              f"R2_drive_OLS_λ=0={sub['R2_drive_lambda0_ols'].mean():.6f}, "
              f"SR_E2E={sr_e2e_cond:.4f}, "
              f"SR_OLS_λ=0={sub['SR_lambda0_ols'].mean():.4f}")
    # Paired test: average across conditions per session first (not treating
    # Tracking/Playback from same session as independent)
    common = merged_abl.dropna(subset=["R2_drive_rollout", "R2_drive_lambda0_ols"])
    session_avg = common.groupby("Session_Idx")[
        ["R2_drive_rollout", "R2_drive_lambda0_ols"]].mean()
    if len(session_avg) > 1:
        t_a, p_a = ttest_rel(session_avg["R2_drive_rollout"],
                             session_avg["R2_drive_lambda0_ols"])
        print(f"  Paired R2_drive (E2E vs OLS_λ=0, per-session avg): "
              f"t={t_a:.3f}, p={p_a:.4f}")
        if p_a < 0.05:
            print(f"  -> Dynamics constraint significantly improves R2_drive over")
            print(f"     InfoNCE-only embedding. Rotation is NOT just an InfoNCE artifact.")
        else:
            print(f"  -> No significant difference: InfoNCE alone may produce comparable")
            print(f"     rotational structure. The dynamics constraint adds little.")

# --- Loss curves (first 5 session-seed pairs) ---
fig, axes = plt.subplots(1, 3, figsize=(10, 3))
n_plot = min(5, len(e2e_histories))
for i in range(n_plot):
    hist = e2e_histories[i]
    color = plt.cm.viridis(i / max(1, n_plot - 1))
    axes[0].plot(hist['loss_total'], c=color, alpha=0.7, lw=0.5)
    axes[1].plot(hist['loss_infonce'], c=color, alpha=0.7, lw=0.5)
    axes[2].plot(hist['loss_dyn'], c=color, alpha=0.7, lw=0.5)
axes[0].set_ylabel("Total Loss"); axes[0].set_xlabel("Epoch")
axes[1].set_ylabel("InfoNCE Loss"); axes[1].set_xlabel("Epoch")
axes[2].set_ylabel("Dynamics MSE"); axes[2].set_xlabel("Epoch")
for ax in axes: ax.set_yscale('log')
plt.suptitle(f"Training Curves ({n_train} sessions × {N_SEEDS} seeds, "
             f"{n_plot} shown)", y=1.02, fontsize=9, fontweight="bold")
plt.tight_layout()
for fmt in ["pdf", "png"]:
    plt.savefig(os.path.join(LIE_OUTPUT_DIR, f"E2E_LossCurves.{fmt}"),
                dpi=150, bbox_inches="tight")
plt.show()

# --- Multi-scale R2_drive decay curve ---
fig, ax = plt.subplots(figsize=(5, 3.5))
for cond, color, marker in [("Tracking", "#440154", "o"),
                              ("Playback", "#21918c", "s")]:
    sub = e2e_df[e2e_df["Condition"] == cond]
    means = sub.groupby("Window_Bins")["R2_drive_rollout"].mean()
    sems = sub.groupby("Window_Bins")["R2_drive_rollout"].sem()
    ax.errorbar(means.index * dt * 1000, means.values,
                yerr=sems.values, c=color, marker=marker, ms=4, lw=1,
                label=cond)
ax.set_xlabel("Rollout Horizon (ms)"); ax.set_ylabel("R2_drive (Rollout)")
ax.axhline(0, c='gray', lw=0.5, ls='--'); ax.legend(fontsize=7)
ax.set_title("R2_drive Decay with Rollout Horizon")
plt.tight_layout()
for fmt in ["pdf", "png"]:
    plt.savefig(os.path.join(LIE_OUTPUT_DIR, f"E2E_MultiScale_R2.{fmt}"),
                dpi=150, bbox_inches="tight")
plt.show()

print("End-to-end training complete.")

## 3. Results: Comparison + Controls

Compare end-to-end metrics against baseline two-stage pipeline and Dummy-CEBRA control.

**Controls grid:**
- (a) Dummy-encoder: shuffled-label CEBRA -> `R2_drive_dummy`
- (b) `lambda_dyn=0` ablation: pure InfoNCE without dynamics constraint
- (c) Drive time-shuffle: permute drive on fixed trained encoder
- (d) Embedding dimension sweep: 3/6/8 (set D_LATENT in Cell 0)

**Gate:** `R2_drive_real > R2_drive_dummy` AND `R2_drive_real > R2_drive_drive_shuffle`

In [ ]:
# ============================================================
# Phase 3 -- Results Comparison: Baseline vs End-to-End vs Dummy
# ============================================================
# IMPORTANT: baseline R2_drive (derivative-based OLS) and E2E R2_drive_rollout
# (trajectory-rollout MSE) are DIFFERENT ESTIMATORS of conceptually related
# quantities.  They are NOT directly comparable on a y=x scatter.  Each is
# gated against its OWN null:
#   - Baseline: R2_drive > R2_drive_shuffle (same embedding, shuffled labels)
#   - Baseline: R2_drive > R2_drive_dummy   (dummy-CEBRA embedding)
#   - E2E:     R2_drive_rollout > R2_drive_shuffle (same encoder, shuffled drive)

if e2e_df is not None and baseline_df is not None:
    print("=" * 60)
    print("  Results Comparison")
    print("=" * 60)
    print("  NOTE: baseline R2_drive (derivative OLS) and E2E R2_drive_rollout")
    print("        (trajectory rollout) are different estimators — not directly")
    print("        comparable.  Each is gated against its own null distribution.")
    print()

    # ---- Merge session-level metrics (SR, eig) ----
    # e2e_session_df has one row per session; baseline_df has one per condition.
    # Average baseline SR/eig across conditions per session for fair comparison.
    baseline_session = baseline_df.groupby(
        ["Subject", "Session_Idx", "Headstage"]
    ).agg(SR_baseline=("SR", "mean"),
          Eig_Real_baseline=("Eig_Real_Mean", "mean"),
          Eig_Imag_baseline=("Eig_Imag_Mean", "mean")).reset_index()

    # Use empirical pooled SR (average of TR/PB condition-specific SR) when available
    if 'SR_Tracking' in e2e_session_df.columns and 'SR_Playback' in e2e_session_df.columns:
        e2e_session_df['SR_empirical'] = e2e_session_df[['SR_Tracking','SR_Playback']].mean(axis=1)

    compare_sr = baseline_session.merge(
        e2e_session_df[["Session_Idx", "Headstage", "SR", "Eig_Real_Mean",
                        "Eig_Imag_Mean", "Loss_final"]],
        on=["Session_Idx", "Headstage"],
        suffixes=("", "_e2e"))

    # Add empirical pooled SR if available
    sr_e2e_col = 'SR_empirical' if 'SR_empirical' in e2e_session_df.columns else 'SR'

    print(f"Session-level merge: {len(compare_sr)} sessions")
    print()
    print("--- Skewness Ratio ---")
    print(f"  Baseline SR (OLS post-hoc):    {compare_sr['SR_baseline'].mean():.4f}")
    print(f"  E2E SR (random-drive diag):    {compare_sr['SR'].mean():.4f}")
    if 'SR_empirical' in e2e_session_df.columns:
        compare_sr['SR_empirical'] = e2e_session_df['SR_empirical'].values
        print(f"  E2E SR (empirical pooled):     {compare_sr['SR_empirical'].mean():.4f}")
        t_sr, p_sr = ttest_rel(compare_sr["SR_baseline"], compare_sr["SR_empirical"])
    else:
        t_sr, p_sr = ttest_rel(compare_sr["SR_baseline"], compare_sr["SR"])
    if len(compare_sr) > 1:
        print(f"  Paired t-test (Baseline vs E2E empirical): t={t_sr:.3f}, p={p_sr:.4f}")

    print()
    print("--- Eigenvalues (per-session diagnostics, NOT cross-session averages) ---")
    print(f"  Baseline |Real| (per-session): "
          f"{compare_sr['Eig_Real_baseline'].round(4).tolist()}")
    print(f"  E2E |Real| (per-session):     "
          f"{compare_sr['Eig_Real_Mean'].round(4).tolist()}")
    print(f"  Baseline |Imag| (per-session): "
          f"{compare_sr['Eig_Imag_baseline'].round(4).tolist()}")
    print(f"  E2E |Imag| (per-session):     "
          f"{compare_sr['Eig_Imag_Mean'].round(4).tolist()}")
    print(f"  (E2E eigenvalues are in arbitrary encoder-scale units —")
    print(f"   not comparable across independently-trained sessions.")
    print(f"   Only SR — a scale-invariant ratio — is cross-session comparable.)")

    # ---- Merge per-condition R2_drive (primary horizon only) ----
    # baseline_df has R2_drive per condition; e2e_df has multi-scale R2_drive_rollout.
    # Only the primary (shortest) horizon is used for fair comparison.
    e2e_primary = e2e_df[e2e_df["Window_Bins"] == VAL_ROLLOUT_LENS[0]]
    compare_r2 = baseline_df[["Subject", "Session_Idx", "Headstage", "Condition",
                               "R2_drive"]].merge(
        e2e_primary[["Session_Idx", "Headstage", "Condition", "R2_drive_rollout"]],
        on=["Session_Idx", "Headstage", "Condition"])

    print()
    print("--- R2_drive ---")
    for cond in ["Tracking", "Playback"]:
        sub = compare_r2[compare_r2["Condition"] == cond]
        print(f"  {cond}: R2_drive_baseline={sub['R2_drive'].mean():.6f}, "
              f"R2_drive_e2e={sub['R2_drive_rollout'].mean():.6f}")

    # ---- Gates: each pipeline vs its OWN null ----
    # Baseline gates (derivative-based R2_drive)
    base_shuf = baseline_df[["Session_Idx", "Headstage", "Condition",
                              "R2_drive", "R2_drive_shuffle"]].copy()
    gate_base_shuf = (base_shuf["R2_drive"] >
                       base_shuf["R2_drive_shuffle"]).sum()
    print(f"\n  Gate (Baseline R2_drive > shuffle): {gate_base_shuf}/"
          f"{len(base_shuf)}")

    if dummy_df is not None:
        base_dummy = base_shuf.merge(
            dummy_df[["Session_Idx", "Headstage", "Condition", "R2_drive_dummy"]],
            on=["Session_Idx", "Headstage", "Condition"])
        gate_base_dummy = (base_dummy["R2_drive"] >
                            base_dummy["R2_drive_dummy"]).sum()
        print(f"  Gate (Baseline R2_drive > Dummy-CEBRA): "
              f"{gate_base_dummy}/{len(base_dummy)}")

    # E2E gate: primary horizon only (where shuffle null is computed).
    # Drop rows where either true or shuffle R2_drive is NaN before comparison.
    e2e_gate = e2e_df[e2e_df["Window_Bins"] == VAL_ROLLOUT_LENS[0]][
        ["Session_Idx", "Headstage", "Condition",
         "R2_drive_rollout", "R2_drive_shuffle"]].dropna().copy()
    gate_e2e_shuf = (e2e_gate["R2_drive_rollout"] >
                      e2e_gate["R2_drive_shuffle"]).sum()
    print(f"  Gate (E2E R2_drive > shuffle, {VAL_ROLLOUT_LENS[0]} bins): "
          f"{gate_e2e_shuf}/{len(e2e_gate)}")

    # --- Visualization ---
    fig, axes = plt.subplots(2, 3, figsize=(10, 6))

    # Row 1, col 1-2: SR scatter per session
    axes[0, 0].scatter(compare_sr["SR_baseline"], compare_sr["SR"],
                       c="#440154", s=25, alpha=0.7)
    lims_sr = [0, 1]
    axes[0, 0].plot(lims_sr, lims_sr, '--', c='gray', lw=0.8)
    axes[0, 0].set_xlim(lims_sr); axes[0, 0].set_ylim(lims_sr)
    axes[0, 0].set_xlabel("SR (Baseline)"); axes[0, 0].set_ylabel("SR (E2E)")
    axes[0, 0].set_title("SR: Baseline vs E2E (per session)")
    if len(compare_sr) > 1:
        t, p = ttest_rel(compare_sr["SR_baseline"], compare_sr["SR"])
        axes[0, 0].text(0.05, 0.95, f"t={t:.2f}, p={p:.3f}",
                        transform=axes[0, 0].transAxes, fontsize=6, va='top')

    # Row 1, col 2: SR bar
    sr_bar = pd.DataFrame({
        "Pipeline": ["Baseline", "E2E"],
        "SR": [compare_sr["SR_baseline"].mean(), compare_sr["SR"].mean()],
        "SEM": [compare_sr["SR_baseline"].sem(), compare_sr["SR"].sem()],
    })
    axes[0, 1].bar(["Baseline", "E2E"], sr_bar["SR"],
                   yerr=sr_bar["SEM"], color=["#440154", "#21918c"],
                   capsize=3, width=0.5)
    axes[0, 1].set_title("Mean SR")
    axes[0, 1].set_ylim(0, 1)

    # Row 1, col 3: loss curve
    axes[0, 2].set_title("Final Training Loss")

    # Row 2, col 1-2: R2_drive self-null (each pipeline vs its own null)
    # Baseline: R2_drive true vs shuffle
    base_r2_bar = baseline_df.groupby("Condition")[
        ["R2_drive", "R2_drive_shuffle"]].mean().reset_index()
    base_r2_melt = pd.melt(base_r2_bar, id_vars=["Condition"],
                           value_vars=["R2_drive", "R2_drive_shuffle"],
                           var_name="Type", value_name="R2_drive")
    sns.barplot(data=base_r2_melt, x="Condition", y="R2_drive", hue="Type",
                ax=axes[1, 0],
                palette={"R2_drive": "#440154", "R2_drive_shuffle": "#B2B2B2"})
    axes[1, 0].set_title("Baseline R2_drive\n(true vs shuffle)")
    axes[1, 0].legend(fontsize=5)

    # E2E: R2_drive_rollout true vs drive-shuffle
    e2e_short = e2e_df[e2e_df["Window_Bins"] == VAL_ROLLOUT_LENS[0]]
    e2e_r2_bar = e2e_short.groupby("Condition")[
        ["R2_drive_rollout", "R2_drive_shuffle"]].mean().reset_index()
    e2e_r2_melt = pd.melt(e2e_r2_bar, id_vars=["Condition"],
                          value_vars=["R2_drive_rollout", "R2_drive_shuffle"],
                          var_name="Type", value_name="R2_drive")
    sns.barplot(data=e2e_r2_melt, x="Condition", y="R2_drive", hue="Type",
                ax=axes[1, 1],
                palette={"R2_drive_rollout": "#21918c",
                         "R2_drive_shuffle": "#B2B2B2"})
    axes[1, 1].set_title("E2E R2_drive_rollout\n(true vs drive-shuffle)")
    axes[1, 1].legend(fontsize=5)

    # Row 2, col 3: Gate pass/fail summary
    labels = ['Base\nvs Shuf', 'Base\nvs Dummy', 'E2E\nvs Shuf']
    passes = [gate_base_shuf, gate_base_dummy if dummy_df is not None else 0,
              gate_e2e_shuf]
    totals = [len(base_shuf), len(base_dummy) if dummy_df is not None else 0,
              len(e2e_gate)]
    colors_bar = ['#440154', '#440154', '#21918c']
    axes[1, 2].bar(labels, [p/max(t,1) for p, t in zip(passes, totals)],
                   color=colors_bar, alpha=0.7)
    for i, (p, t) in enumerate(zip(passes, totals)):
        if t > 0:
            axes[1, 2].text(i, p/t + 0.02, f'{p}/{t}', ha='center', fontsize=7)
    axes[1, 2].set_ylabel("Pass Fraction"); axes[1, 2].set_ylim(0, 1.1)
    axes[1, 2].set_title("Gate Pass Rates")

    plt.suptitle("Baseline vs End-to-End Lie Dynamics Comparison",
                 y=1.02, fontsize=10, fontweight="bold")
    plt.tight_layout()
    for fmt in ["pdf", "png"]:
        plt.savefig(os.path.join(LIE_OUTPUT_DIR, f"E2E_vs_Baseline.{fmt}"),
                    dpi=150, bbox_inches="tight")
    plt.show()

else:
    print("Skipped comparison: missing data (e2e or baseline).")

In [ ]:
# ============================================================
# Phase 4 -- Tracking vs Playback Paired Analysis
# ============================================================
# Note: SR/eigenvalues are from the shared session-level generator J(u)+L,
# so Tracking/PB comparisons are only meaningful for R2_drive (held-out epochs).
# SR/eig are reported as session-level summaries (pooled across conditions).
#
# CAVEAT -- kinematic confound: drive is standardized across pooled TR+PB epochs.
# If the animal moves less during Playback, the drive dynamic range is smaller,
# which may systematically lower R2_drive_rollout for Playback INDEPENDENTLY of
# neural computation.  Observed TR > PB differences cannot be attributed to
# neural mechanisms unless velocity distributions are first shown comparable
# (cf. lie_algebra_method_description.md section 12.9).

# ---- Velocity distribution check (kinematic confound) ----
print("=" * 60)
print("  Velocity Distribution Check (Kinematic Confound)")
print("=" * 60)

vel_stats = []
for idx, (n_data_session, f_df) in enumerate(zip(n_data_all, f_data_all)):
    for val, label in [(0.0, "Tracking"), (1.0, "Playback")]:
        v = f_df[f_df["Condition"] == val]["Velocity_x"].values
        vel_stats.append({
            "Session_Idx": idx, "Condition": label,
            "Mean": np.mean(v), "Std": np.std(v),
            "RMS": np.sqrt(np.mean(v**2)),
            "Range": np.ptp(v),
            "N_tp": len(v),
        })
vel_df = pd.DataFrame(vel_stats)
print(vel_df.groupby("Condition")[["Mean", "Std", "RMS", "Range"]].mean().round(2).to_string())
print()

# Paired t-test on velocity metrics
vel_pivot = vel_df.pivot_table(
    values=["Mean", "Std", "RMS", "Range"],
    index="Session_Idx", columns="Condition").dropna()
if len(vel_pivot) > 1:
    print("  Tracking vs Playback velocity paired t-tests:")
    for metric in ["RMS", "Std", "Range"]:
        t, p = ttest_rel(vel_pivot[metric]["Tracking"],
                         vel_pivot[metric]["Playback"])
        print(f"    {metric:10s}: TR={vel_pivot[metric]['Tracking'].mean():.2f}, "
              f"PB={vel_pivot[metric]['Playback'].mean():.2f}, "
              f"t={t:.3f}, p={p:.4f}")
    # Per-session KS test (formal distribution comparison)
    from scipy.stats import ks_2samp
    ks_results = []
    for idx, f_df in enumerate(f_data_all):
        v_tr = f_df[f_df["Condition"] == 0.0]["Velocity_x"].values
        v_pb = f_df[f_df["Condition"] == 1.0]["Velocity_x"].values
        if len(v_tr) > 10 and len(v_pb) > 10:
            ks_stat, ks_p = ks_2samp(v_tr, v_pb)
            ks_results.append({"Session_Idx": idx, "KS_stat": ks_stat, "KS_p": ks_p})
    if ks_results:
        ks_df = pd.DataFrame(ks_results)
        n_sig = (ks_df["KS_p"] < 0.05).sum()
        print(f"  KS test (Tracking vs Playback per session):")
        print(f"    Sessions with p<0.05: {n_sig}/{len(ks_df)}")
        print(f"    Mean KS stat: {ks_df['KS_stat'].mean():.3f}, "
              f"median p: {ks_df['KS_p'].median():.3f}")
        if n_sig > len(ks_df) / 2:
            print(f"    WARNING: majority of sessions show significantly")
            print(f"      different velocity distributions -- kinematic confound")
            print(f"      is likely.")
    print()

# Velocity histogram
fig, axes = plt.subplots(1, 2, figsize=(6, 2.5))
for i, (cond, color) in enumerate([("Tracking", "#440154"), ("Playback", "#21918c")]):
    all_v = np.concatenate([
        f_df[f_df["Condition"] == (0.0 if cond == "Tracking" else 1.0)]["Velocity_x"].values
        for f_df in f_data_all])
    axes[i].hist(all_v, bins=50, color=color, alpha=0.7, density=True)
    axes[i].set_title(f"{cond}\n(RMS={np.sqrt(np.mean(all_v**2)):.1f}, "
                      f"N={len(all_v):,} tp)")
    axes[i].set_xlabel("Velocity_x")
axes[0].set_ylabel("Density")
plt.suptitle("Velocity Distribution: Tracking vs Playback (all sessions)",
             y=1.05, fontsize=9, fontweight="bold")
plt.tight_layout()
for fmt in ["pdf", "png"]:
    plt.savefig(os.path.join(LIE_OUTPUT_DIR, f"Velocity_Distribution_TR_PB.{fmt}"),
                dpi=150, bbox_inches="tight")
plt.show()

# ---- Per-condition SR (condition-specific drive distribution) ----
if 'e2e_session_df' in dir() and e2e_session_df is not None:
    if 'SR_Tracking' in e2e_session_df.columns and 'SR_Playback' in e2e_session_df.columns:
        print("  Per-condition SR (drive-distribution-specific):")
        sr_tr = e2e_session_df['SR_Tracking'].dropna()
        sr_pb = e2e_session_df['SR_Playback'].dropna()
        # Align by session
        common = e2e_session_df[['Session_Idx', 'SR_Tracking', 'SR_Playback']].dropna()
        print(f"    SR_Tracking: {common['SR_Tracking'].mean():.4f} +- {common['SR_Tracking'].sem():.4f}")
        print(f"    SR_Playback:  {common['SR_Playback'].mean():.4f} +- {common['SR_Playback'].sem():.4f}")
        if len(common) > 1:
            t_sr, p_sr = ttest_rel(common['SR_Tracking'], common['SR_Playback'])
            print(f"    Paired t-test: t={t_sr:.3f}, p={p_sr:.4f}")
            if p_sr < 0.05:
                print(f"    -> Tracking SR significantly higher: rotation specifically")
                print(f"       enhanced, not just global gain modulation.")
        print()

if e2e_df is not None:
    print("=" * 60)
    print("  Tracking vs Playback -- Paired Comparison")
    print("=" * 60)

    # ---- Session-level SR/eig (pooled across conditions) ----
    if 'e2e_session_df' in dir() and e2e_session_df is not None:
        print("--- Session-level Generator Metrics (shared J(u)+L) ---")
        print(f"  Mean SR:           {e2e_session_df['SR'].mean():.4f} "
              f"(sem={e2e_session_df['SR'].sem():.4f})")
        print(f"  N sessions:        {len(e2e_session_df)}")
        print(f"  (Eigenvalues are per-session diagnostics in arbitrary encoder-scale")
        print(f"   units — not comparable across independently-trained sessions.")
        print(f"   See per-session table in Cell 10 for individual values.)")
        print()

    # ---- Per-condition R2_drive: Tracking vs Playback ----
    # Stratify by Window_Bins: each horizon gets its own paired test
    # Per-horizon paired t-tests
    for wlen in VAL_ROLLOUT_LENS:
        sub = e2e_df[e2e_df["Window_Bins"] == wlen]
        pivot_e2e = sub.pivot_table(
            values=["R2_drive_rollout"],
            index=["Subject", "Session_Idx", "Headstage"],
            columns="Condition").dropna()
        if len(pivot_e2e) > 1:
            print(f"    {wlen} bins ({wlen*dt*1000:.0f}ms): ", end="")
            tr = pivot_e2e["R2_drive_rollout"]["Tracking"].values
            pb = pivot_e2e["R2_drive_rollout"]["Playback"].values
            t, p = ttest_rel(tr, pb)
            print(f"TR={np.mean(tr):.6f}, PB={np.mean(pb):.6f}, "
                  f"t={t:.3f}, p={p:.4f}")
        else:
            print(f"    {wlen} bins: not enough paired sessions")

    # ---- Visualization (primary horizon = shortest window) ----
    fig, axes = plt.subplots(1, 2, figsize=(6, 3))
    sub_primary = e2e_df[e2e_df["Window_Bins"] == VAL_ROLLOUT_LENS[0]]
    pivot_primary = sub_primary.pivot_table(
        values=["R2_drive_rollout"],
        index=["Subject", "Session_Idx", "Headstage"],
        columns="Condition").dropna()

    # Panel A: Session-level SR histogram
    if 'e2e_session_df' in dir() and e2e_session_df is not None:
        axes[0].hist(e2e_session_df["SR"], bins=min(10, len(e2e_session_df)),
                     color="#440154", alpha=0.7, edgecolor='white')
        axes[0].axvline(e2e_session_df["SR"].mean(), color='#21918c',
                        lw=1.5, ls='--', label=f'Mean={e2e_session_df["SR"].mean():.3f}')
        axes[0].set_xlabel("Skewness Ratio"); axes[0].set_ylabel("Sessions")
        axes[0].set_title("SR Distribution (E2E)")
        axes[0].legend(fontsize=5)

    # Panel B: R2_drive Tracking vs Playback paired (primary horizon)
    if len(pivot_primary) > 1:
        tr_vals = pivot_primary["R2_drive_rollout"]["Tracking"].values
        pb_vals = pivot_primary["R2_drive_rollout"]["Playback"].values
        for i in range(len(tr_vals)):
            axes[1].plot([0, 1], [tr_vals[i], pb_vals[i]], '-',
                         c='gray', lw=0.4, alpha=0.5)
        axes[1].scatter(np.zeros(len(tr_vals)), tr_vals, c="#440154",
                        s=30, zorder=3, label="Tracking")
        axes[1].scatter(np.ones(len(pb_vals)), pb_vals, c="#21918c",
                        s=30, zorder=3, label="Playback")
        axes[1].set_xticks([0, 1])
        axes[1].set_xticklabels(["Tracking", "Playback"], fontsize=7)
        axes[1].set_ylabel("R2_drive (Rollout)")
        axes[1].set_title("R2_drive: TR vs PB")
        axes[1].legend(fontsize=5)

    plt.suptitle("End-to-End Lie Dynamics -- Session Metrics",
                 y=1.02, fontsize=9, fontweight="bold")
    plt.tight_layout()
    for fmt in ["pdf", "png"]:
        plt.savefig(os.path.join(LIE_OUTPUT_DIR,
                                 f"E2E_Tracking_vs_Playback.{fmt}"),
                    dpi=150, bbox_inches="tight")
    plt.show()

else:
    print("Skipped: no E2E data available.")

In [ ]:
# ============================================================
# Summary Output
# ============================================================
# Write timestamped summary .txt with all metrics.
# Mirrors the output style of Skieur_LieAlgebra_CEBRA.ipynb Cell 19.

if 'LIE_OUTPUT_DIR' not in dir():
    LIE_OUTPUT_DIR = f"Skieur_LieE2E_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}"
    os.makedirs(LIE_OUTPUT_DIR, exist_ok=True)

out_path = os.path.join(LIE_OUTPUT_DIR, "LieE2E_summary.txt")

with open(out_path, "w") as f:
    f.write("=" * 70 + "\n")
    f.write("  SKIEUR End-to-End Lie Dynamics -- Summary Report\n")
    f.write("=" * 70 + "\n")
    f.write(f"  Generated: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write("\n")
    f.write("--- Parameters ---\n")
    f.write(f"  dt                  = {dt}\n")
    f.write(f"  SESSION_TYPE        = {SESSION_TYPE}\n")
    f.write(f"  USE_MACRO_EPOCH     = {USE_MACRO_EPOCH}\n")
    f.write(f"  MIN_EPOCH_DUR       = {MIN_EPOCH_DUR} s\n")
    f.write(f"  CEBRA_DISTANCE      = {CEBRA_DISTANCE}\n")
    f.write(f"  CEBRA_ARCH          = {CEBRA_ARCH}\n")
    f.write(f"  CEBRA_EMBEDDING_DIM = {CEBRA_EMBEDDING_DIM}\n")
    f.write(f"  TAU_SHIFT           = {TAU_SHIFT} bins\n")
    f.write(f"  LIE_METHOD          = {LIE_METHOD}\n")
    f.write("\n")
    f.write("--- End-to-End Config ---\n")
    f.write(f"  D_LATENT            = {D_LATENT}\n")
    f.write(f"  USE_ODE             = {USE_ODE}\n")
    f.write(f"  ODE_METHOD          = {ODE_METHOD}\n")
    f.write(f"  LAMBDA_DYN          = {LAMBDA_DYN}\n")
    f.write(f"  LAMBDA_DYN_WARMUP   = {LAMBDA_DYN_WARMUP}\n")
    f.write(f"  CONSTRAINED_L       = {CONSTRAINED_L}\n")
    f.write(f"  TEMPERATURE         = {TEMPERATURE}\n")
    f.write(f"  DRIVE_KEYS          = {DRIVE_KEYS}\n")
    f.write(f"  MINI_TRAJ_LEN       = {MINI_TRAJ_LEN}\n")
    f.write(f"  VAL_ROLLOUT_LENS    = {VAL_ROLLOUT_LENS}\n")
    f.write(f"  N_EPOCHS_TRAIN      = {N_EPOCHS_TRAIN}\n")
    f.write(f"  BATCH_SIZE          = {BATCH_SIZE}\n")
    f.write(f"  LR                  = {LR}\n")
    f.write(f"  N_SHUFFLES          = {N_SHUFFLES}\n")
    f.write(f"  N_SEEDS             = {N_SEEDS}\n")
    f.write(f"  RANDOM_SEED         = {RANDOM_SEED}\n")
    f.write(f"  TRAIN_VAL_SPLIT     = {TRAIN_VAL_SPLIT}\n")
    f.write(f"  DEVICE              = {DEVICE}\n")
    f.write("\n")
    f.write("--- Sessions ---\n")
    f.write(f"  Total sessions     : {len(n_data_all)}\n")
    f.write(f"  Headstage 0        : {n_hs0}\n")
    f.write(f"  Headstage 1        : {len(n_data_all) - n_hs0}\n")
    try:
        f.write(f"  Sessions trained   : {n_train}\n")
    except NameError:
        f.write(f"  Sessions trained   : {N_TRAIN_SESSIONS if N_TRAIN_SESSIONS is not None else 'all'}\n")
    f.write("\n")

    # --- Baseline results ---
    f.write("=" * 70 + "\n")
    f.write("  1. Baseline Two-Stage CEBRA-Embedded Lie\n")
    f.write("=" * 70 + "\n")
    try:
        if baseline_df is not None:
            grp = baseline_df.groupby("Condition")[
                ["SR", "R2", "R2_drive", "SR_shuffle", "R2_drive_shuffle",
                 "Eig_Real_Mean", "Eig_Imag_Mean"]].mean().round(4)
            f.write(grp.to_string() + "\n\n")
    except NameError:
        f.write("  (Not run)\n\n")

    # --- Dummy CEBRA ---
    f.write("=" * 70 + "\n")
    f.write("  2. Dummy-CEBRA Negative Control\n")
    f.write("=" * 70 + "\n")
    try:
        if dummy_df is not None:
            merged = baseline_df.merge(dummy_df,
                                       on=["Subject", "Session_Idx",
                                           "Headstage", "Condition"])
            for cond in ["Tracking", "Playback"]:
                sub = merged[merged["Condition"] == cond]
                f.write(f"  {cond}: SR_true={sub['SR'].mean():.4f}, "
                        f"SR_dummy={sub['SR_dummy'].mean():.4f}, "
                        f"R2_drive_true={sub['R2_drive'].mean():.6f}, "
                        f"R2_drive_dummy={sub['R2_drive_dummy'].mean():.6f}\n")
            gate = (merged["R2_drive"] > merged["R2_drive_dummy"]).sum()
            f.write(f"  R2_drive gate passed: {gate}/{len(merged)}\n\n")
        else:
            f.write("  (Not run)\n\n")
    except NameError:
        f.write("  (Not run)\n\n")

    # --- End-to-End results ---
    f.write("=" * 70 + "\n")
    f.write("  3. End-to-End Lie Dynamics\n")
    f.write("=" * 70 + "\n")
    try:
        if 'e2e_session_df' in dir() and e2e_session_df is not None:
            f.write("--- Session-level Generator Metrics ---\n")
            f.write(f"  N sessions: {len(e2e_session_df)}\n")
            f.write(f"  SR (random-drive diagnostic, N(0,1)): "
                    f"{e2e_session_df['SR'].mean():.4f} "
                    f"sem={e2e_session_df['SR'].sem():.4f}\n")
            if 'SR_Tracking' in e2e_session_df.columns:
                f.write(f"  SR_Tracking (empirical drive):   "
                        f"{e2e_session_df['SR_Tracking'].mean():.4f} "
                        f"sem={e2e_session_df['SR_Tracking'].sem():.4f}\n")
                f.write(f"  SR_Playback (empirical drive):   "
                        f"{e2e_session_df['SR_Playback'].mean():.4f} "
                        f"sem={e2e_session_df['SR_Playback'].sem():.4f}\n")
                common_sr = e2e_session_df[['SR_Tracking','SR_Playback']].dropna()
                if len(common_sr) > 1:
                    t_sr, p_sr = ttest_rel(common_sr['SR_Tracking'],
                                           common_sr['SR_Playback'])
                    f.write(f"  SR TR vs PB paired t-test: t={t_sr:.3f}, p={p_sr:.4f}\n")
            f.write(f"  (Eigenvalues are per-session diagnostics — not cross-session comparable)\n")
            f.write(f"  Per-session |Real|: "
                    f"{e2e_session_df['Eig_Real_Mean'].round(4).tolist()}\n")
            f.write(f"  Per-session |Imag|: "
                    f"{e2e_session_df['Eig_Imag_Mean'].round(4).tolist()}\n")
            f.write("\n")
        if e2e_df is not None:
            f.write("--- Multi-scale R2_drive (held-out rollout) ---\n")
            for wlen in VAL_ROLLOUT_LENS:
                sub = e2e_df[e2e_df["Window_Bins"] == wlen]
                grp = sub.groupby("Condition")["R2_drive_rollout"].mean().round(6)
                f.write(f"  {wlen} bins ({wlen*dt*1000:.0f}ms):\n")
                f.write(f"    {grp.to_string()}\n")
            # Primary-horizon paired test
            sub_p = e2e_df[e2e_df["Window_Bins"] == VAL_ROLLOUT_LENS[0]]
            pivot_e2e = sub_p.pivot_table(
                values=["R2_drive_rollout"],
                index=["Subject", "Session_Idx", "Headstage"],
                columns="Condition").dropna()
            if len(pivot_e2e) > 1:
                f.write("  Tracking vs Playback (primary horizon):\n")
                t, p = ttest_rel(pivot_e2e["R2_drive_rollout"]["Tracking"],
                                 pivot_e2e["R2_drive_rollout"]["Playback"])
                f.write(f"    R2_drive_rollout: t={t:.3f}, p={p:.4f}\n")
            f.write("\n")
        # Ablation
        if ablation_df is not None:
            f.write("--- lambda_dyn=0 Ablation (post-hoc OLS on frozen embedding) ---\n")
            for cond in ["Tracking", "Playback"]:
                sub_a = ablation_df[ablation_df["Condition"] == cond]
                f.write(f"  {cond}: SR_OLS={sub_a['SR_lambda0_ols'].mean():.4f}, "
                        f"R2_drive_OLS={sub_a['R2_drive_lambda0_ols'].mean():.6f}\n")
            f.write("\n")
        # Velocity confound
        try:
            if 'vel_df' in dir():
                f.write("--- Velocity Distribution Check ---\n")
                f.write(vel_df.groupby("Condition")[["RMS","Std","Range"]].mean().round(2).to_string())
                f.write("\n\n")
        except NameError:
            pass
    except NameError:
        f.write("  (Not run)\n\n")

    f.write("=" * 70 + "\n")
    f.write("  End of Report\n")
    f.write("=" * 70 + "\n")

print(f"Summary written to: {out_path}")
print(f"All outputs in: {LIE_OUTPUT_DIR}")

# --- Final cleanup ---
import gc
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("Done.")